# AFES — Statistical Arbitrage Pipeline

End-to-end notebook: data download → pair selection → filters & sizing → backtest → analysis.
Variables computed in Step 4 (`df_trades`, `pair_results`, `closes`, `pairs`) remain in memory for Step 5.


## 0. Setup


In [ ]:
import sys, os
from pathlib import Path

# Ensure project root is on path so all local modules resolve
PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

%matplotlib inline
matplotlib.rcParams["figure.dpi"] = 100

print(f"Project root : {PROJECT_ROOT}")
print(f"Python       : {sys.version.split()[0]}")
print(f"Pandas       : {pd.__version__}")
print(f"NumPy        : {np.__version__}")


## Step 1 — Download Data

> **Skip** this cell if CSVs already exist in `data/`.


In [ ]:
%run step1a_download.py


## Step 2 — Pair Selection


In [ ]:
%run step2a_pairs.py


In [ ]:
import pandas as pd
from config import DATA_DIR
_pairs_preview = pd.read_csv(DATA_DIR / "pairs_selected.csv")
print(f"{len(_pairs_preview)} pairs selected")
display(_pairs_preview[["pair", "beta_daily", "half_life_bars", "hurst", "correlation", "coint_pvalue_daily"]].head(20))


## Step 3 — Filters & Sizing


### 3a — HMM Regime Detection


In [ ]:
%run step3a_hmm.py


### 3b — K-Means Macro Regime


In [ ]:
%run step3b_kmeans.py


### 3c — Implied Volatility Filter


In [ ]:
%run step3c_iv.py


### 3d — OU Monte Carlo


In [ ]:
%run step3d_montecarlo.py


### 3e — Dynamic Position Sizing


In [ ]:
from step3e_sizing import main as _sizing_main
_sizing_main()


### 3f — Per-Pair Parameter Grid


In [ ]:
%run step3f_grid.py


### 3g — Regime Profiler (RCDP)


In [ ]:
%run step3g_regime_profiler.py


### 3h — Z-Score Profiler


In [ ]:
%run step3h_z_profiler.py


### 3i — Profiler


In [ ]:
%run step3i_profiler.py


### 3j — Walk-Forward Optimisation


In [ ]:
%run step3j_wfo.py


## Step 4 — Backtest

Runs the full backtest inline so `df_trades`, `pair_results`, `closes`, and `pairs` remain in memory for Step 5.


In [ ]:
# ── Step 4: Inline backtest — df_trades and pair_results stay in memory ──
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from config import (
    ENTRY_Z, EXIT_Z, STOP_Z, ENTRY_Z_VOLATILE,
    COST_MAKER, COST_TAKER, CIRCUIT_BREAKER_Z, BORROW_RATE_ANNUAL, PAIR_MAX_LOSS,
    IV_SIZE_NORM, MIN_POSITION_SIZE, TRAIN_RATIO,
    RTH_START, RTH_END, SIGNAL_START, RECENT_BARS,
    DATA_DIR, OUTPUT_DIR, INITIAL_CAPITAL,
    ALLOCATION_METHOD, MAX_PAIR_WEIGHT, TARGET_RISK_USD,
    BARS_PER_DAY, CLOSES_FILE, KALMAN_DELTA,
    COINT_WINDOW_DAYS, COINT_BREAK_P, COINT_RECHECK_DAYS,
    RVOL_MIN_ENTRY, VELOCITY_BARS, VWAP_BAND_SIGMA,
    TAIL_EV_GATE, TAIL_ENTRY_Z_MIN, TAIL_STOP_Z, TAIL_EXIT_Z,
    TAIL_LABEL_LOOKAHEAD_BARS, TAIL_THRESHOLD, TAIL_RR_THRESHOLD,
    TAIL_CONFIDENCE_LEVEL, TAIL_REFIT_FREQ,
    PORTFOLIO_OPT_MAX_GROSS, PORTFOLIO_OPT_WEIGHT_MIN, PORTFOLIO_OPT_WEIGHT_MAX,
)
from kalman import kalman_hedge
from step3e_sizing import (
    load_regimes, load_iv, load_global_hmm, load_mc_confidence,
    load_corr_throttle, load_hrp_weights,
    iv_multiplier_series, position_size,
)
from filters import (
    CointegrationFilter, MacroFilter, HurstFilter,
    validate_kde_density, BreakVelocityDetector
)
from config import HURST_ENTRY_WINDOW
from tail_ev_profiler import TailAdjustedEVProfiler
from regime_memory import RegimeMemoryWeighter

try:
    from rmt_covariance import clean_covariance_rmt
    from portfolio_optimizer import RegularizedPortfolioOptimizer
    from portfolio_margin_risk import MarginSpiralDetector
except Exception:
    clean_covariance_rmt = None
    RegularizedPortfolioOptimizer = None
    MarginSpiralDetector = None

BARS_PER_TRADING_DAY = BARS_PER_DAY
def _data_path() -> str:
    p = DATA_DIR / CLOSES_FILE
    if not p.exists():
        fallback = DATA_DIR / "closes_15min.csv"
        if fallback.exists():
            return str(fallback)
        raise FileNotFoundError(f"No data file: {CLOSES_FILE}")
    return str(p)


def load_closes() -> pd.DataFrame:
    """Load ALL available intraday data.

    Pair selection is done on 8-year daily data (step2), so there is no
    look-ahead: using the full intraday history is valid and maximises
    the number of trades for statistical evaluation.
    """
    closes = pd.read_csv(_data_path(), index_col=0, parse_dates=True)
    closes.index = pd.to_datetime(closes.index, utc=True).tz_convert("US/Eastern")
    closes = closes.between_time(RTH_START, RTH_END)

    pairs_path = DATA_DIR / "pairs_selected.csv"
    if pairs_path.exists():
        meta = pd.read_csv(pairs_path)
        if not meta.empty:
            needed    = {t for p in meta["pair"] for t in p.split("-")}
            available = [t for t in needed if t in closes.columns]
            # No global dropna — each pair drops its own NaNs in build_signals
            return closes[available]

    return closes


def build_signals(closes, t1, t2, beta, half_life,
                  volumes: pd.DataFrame | None = None,
                  delta: float = KALMAN_DELTA) -> pd.DataFrame:
    """Kalman-filter spread signals with optional VW-Z / RVOL / VWAP columns.

    When `volumes` is provided (volumes_Nmin.csv), three extra columns are added:
      vwz         — volume-weighted z-score (primary entry signal)
      rvol        — relative volume vs 20-day rolling avg at same time-of-day
      vwap_spread — intraday volume-weighted anchor of the spread (resets daily)

    When volumes is None all three columns are absent and the backtest falls back
    to the regular zscore for entry decisions.
    """
    # Drop rows where either leg has NaN (e.g., pre-IPO) before passing to Kalman
    pair_closes = closes[[t1, t2]].dropna()
    alpha_arr, beta_arr, innov, _ = kalman_hedge(
        pair_closes[t1].values, pair_closes[t2].values,
        delta=delta, beta_init=float(beta),
    )
    spread  = pd.Series(innov,    index=pair_closes.index, name="spread")
    beta_s  = pd.Series(beta_arr, index=pair_closes.index, name="beta")
    alpha_s = pd.Series(alpha_arr, index=pair_closes.index, name="alpha")
    window  = max(20, min(int(half_life), 200))
    spread_std = spread.rolling(window=window).std()
    zscore     = spread / spread_std

    df = pd.DataFrame({
        f"{t1}_close": pair_closes[t1],
        f"{t2}_close": pair_closes[t2],
        "spread":      spread,
        "beta":        beta_s,
        "alpha":       alpha_s,
        "innov":       innov,
        "spread_std":  spread_std,
        "zscore":      zscore,
    }).dropna().between_time(SIGNAL_START, RTH_END)
    df["velocity"] = df["zscore"].diff(VELOCITY_BARS)

    # ── M15 Kalman z-score: signal on 15-min, execution on 5-min ─────────────
    # Resample pair closes to 15-min, run separate Kalman, ffill back to 5-min.
    # is_m15_close marks the last 5-min bar of each 15-min period (exit gate).
    try:
        pc15 = pair_closes[[t1, t2]].resample("15min").last().dropna()
        if len(pc15) >= 40:
            a15, b15, innov15, _ = kalman_hedge(
                pc15[t1].values, pc15[t2].values,
                delta=delta, beta_init=float(beta),
            )
            sp15  = pd.Series(innov15, index=pc15.index)
            win15 = max(10, min(int(half_life // 3), 67))  # 15-min bars ≈ 5-min/3
            std15 = sp15.rolling(win15).std()
            z15   = (sp15 / std15).dropna()
            # Forward-fill 15-min z onto 5-min index (no look-ahead)
            z15_ff = z15.reindex(df.index, method="ffill")
            df["z_m15"] = z15_ff
            # Mark last 5-min bar of each 15-min group
            m15_closes = set(pc15.index)
            df["is_m15_close"] = df.index.isin(m15_closes)
        else:
            df["z_m15"]       = df["zscore"]
            df["is_m15_close"] = True
    except Exception:
        df["z_m15"]       = df["zscore"]
        df["is_m15_close"] = True

    # Pre-calculate Break Velocity components
    df["half_life_bars"] = half_life
    nu_s = df["innov"]
    var_nu_s = nu_s.rolling(60).var().bfill()
    beta_s = df["beta"]
    d_beta_dt_s = beta_s.diff().abs().rolling(10).mean().fillna(0)
    hl_s = df["half_life_bars"]
    hl_median_s = hl_s.rolling(300).median().bfill()
    
    # Toxicity Filter (Hawkes)
    from toxicity import HawkesToxicityFilter
    toxicity_f = HawkesToxicityFilter()
    df["toxicity"] = toxicity_f.compute_intensity(df["zscore"].diff())

    # Initialize Detectors
    bv_detector = BreakVelocityDetector(threshold=4.5)
    ev_profiler = TailAdjustedEVProfiler(u_threshold=3.0)
    ev_profiler.fit_tail(f"{t1}-{t2}", df["zscore"]) # Fit GPD to historical tail
    
    df["break_score"] = 0.0
    df["is_broken"]   = False
    df["tail_ev"]     = 0.0

    for i, (ts, row_data) in enumerate(df.iterrows()):
        bt, is_broken = bv_detector.get_break_score(
            nu_t=row_data["innov"],
            rolling_var_nu=var_nu_s.iloc[i],
            hl_t=hl_s.iloc[i],
            hl_median=hl_median_s.iloc[i],
            d_beta_dt=d_beta_dt_s.iloc[i],
            z_t=row_data["zscore"]
        )
        df.loc[ts, "break_score"] = bt
        df.loc[ts, "is_broken"]   = is_broken

    if volumes is not None and t1 in volumes.columns and t2 in volumes.columns:
        v1  = volumes[t1].reindex(df.index).fillna(0)
        v2  = volumes[t2].reindex(df.index).fillna(0)
        vol = (v1 + abs(beta) * v2).clip(lower=1.0)

        # ── VW-Z: volume-weighted z-score ────────────────────────────────────
        # High-volume bars anchor the "fair value" of the spread more strongly
        # than quiet bars, so the resulting z-score filters out thin-market noise.
        vw_sum  = vol.rolling(window).sum().clip(lower=1.0)
        vw_mean = (df["spread"] * vol).rolling(window).sum() / vw_sum
        vw_var  = (vol * (df["spread"] - vw_mean) ** 2).rolling(window).sum() / vw_sum
        df["vwz"] = (df["spread"] - vw_mean) / np.sqrt(vw_var.clip(lower=1e-16))

        # ── RVOL: volume relative to same time-of-day 20-day rolling avg ─────
        # shift(1) inside the transform makes each bar's reference use only
        # the PREVIOUS day's rolling average — no lookahead.
        tod_avg = vol.groupby(vol.index.time).transform(
            lambda x: x.rolling(20, min_periods=5).mean().shift(1)
        ).clip(lower=1.0)
        df["rvol"] = vol / tod_avg

        # ── Intraday spread VWAP (resets at session open each day) ───────────
        dates         = df.index.normalize()
        cum_sv        = (df["spread"] * vol).groupby(dates).cumsum()
        cum_v         = vol.groupby(dates).cumsum().clip(lower=1.0)
        df["vwap_spread"] = cum_sv / cum_v

        # ── M15 VWAP + σ-bands (period-anchored, resets every 3 bars = 15 min) ─
        # Each 15-min candle group (9:30-9:44, 9:45-9:59, …) gets its own
        # cumulative VWAP and volume-weighted variance.
        # z_m15 = (spread - vwap_m15) / std_m15  →  ±1/2/3σ bands
        m15_key = pd.Series(
            (df.index.hour * 4 + df.index.minute // 15),
            index=df.index,
        ).astype(str) + "_" + df.index.normalize().astype(str)
        cum_sv_m15  = (df["spread"] * vol).groupby(m15_key).cumsum()
        cum_v_m15   = vol.groupby(m15_key).cumsum().clip(lower=1.0)
        vwap_m15    = cum_sv_m15 / cum_v_m15
        cum_sv2_m15 = (vol * (df["spread"] - vwap_m15) ** 2).groupby(m15_key).cumsum()
        std_m15     = np.sqrt((cum_sv2_m15 / cum_v_m15).clip(lower=0))
        df["vwap_m15"]     = vwap_m15
        df["vwap_std_m15"] = std_m15
        df["vwap_z_m15"]   = (df["spread"] - vwap_m15) / std_m15.clip(lower=1e-10)

        # ── H1 VWAP + σ-bands (period-anchored, resets every 12 bars = 60 min) ─
        h1_key = pd.Series(
            df.index.hour.astype(str),
            index=df.index,
        ) + "_" + df.index.normalize().astype(str)
        cum_sv_h1  = (df["spread"] * vol).groupby(h1_key).cumsum()
        cum_v_h1   = vol.groupby(h1_key).cumsum().clip(lower=1.0)
        vwap_h1    = cum_sv_h1 / cum_v_h1
        cum_sv2_h1 = (vol * (df["spread"] - vwap_h1) ** 2).groupby(h1_key).cumsum()
        std_h1     = np.sqrt((cum_sv2_h1 / cum_v_h1).clip(lower=0))
        df["vwap_h1"]     = vwap_h1
        df["vwap_std_h1"] = std_h1
        df["vwap_z_h1"]   = (df["spread"] - vwap_h1) / std_h1.clip(lower=1e-10)

    return df




def load_kmeans_regime() -> pd.Series | None:
    """Load daily K-Means macro regime labels (0=Trend, 1=Sideways, 2=Panic)."""
    path = DATA_DIR / "kmeans_regimes.csv"
    if not path.exists():
        return None
    s = pd.read_csv(path, index_col=0, parse_dates=True).iloc[:, 0]
    if s.index.tz is None:
        s.index = s.index.tz_localize("UTC")
    return s


def min_viable_entry_z(sigma_spread: float, avg_notional: float,
                        exit_z: float, safety: float = 2.5) -> float:
    """
    Minimum entry Z-score for expected gross P&L to exceed transaction costs.

    Expected gross per trade = (entry_z + |exit_z|) × sigma_spread
    Break-even condition:
        (entry_z + |exit_z|) × sigma >= safety × 2 × COST_MAKER, COST_TAKER, CIRCUIT_BREAKER_Z × notional
        → entry_z_min = safety × cost_fraction − |exit_z|

    safety=2.5 means expected profit must be 2.5× the transaction cost.
    If the grid/MC optimised entry_z is below this floor, we raise it.
    """
    cost_frac = (2 * COST_TAKER * avg_notional) / max(sigma_spread, 1e-8)
    return max(safety * cost_frac - abs(exit_z), 0.0)


def ou_params_from_spread(spread: pd.Series) -> tuple[float, float, float]:
    """OLS on ΔS = a - θ·S_{t-1} + ε  →  returns (theta_per_bar, mu, residual_std)."""
    ds  = spread.diff().dropna()
    lag = spread.shift(1).dropna()
    idx = ds.index.intersection(lag.index)
    
    # Add constant for unbiased theta and mu estimation
    reg = sm.OLS(ds[idx], sm.add_constant(lag[idx])).fit()
    a = float(reg.params.iloc[0])
    b = float(reg.params.iloc[1])
    theta = -b
    mu = a / theta if theta > 0 else spread.mean()
    return max(theta, 1e-6), mu, float(reg.resid.std())


def optimal_thresholds(theta: float, sigma_roll: float, notional: float,
                       n_sim: int = 5000, max_bars: int = 1500
                       ) -> tuple[float, float, float]:
    """
    Joint Monte Carlo grid search for (entry_z, exit_thresh, stop_thresh).

    OU in z-score space: Z_{t+1} = Z_t·(1-θ) + √(2θ)·ε
    LONG: enter at -z_e, exit when Z >= z_x, stop when Z <= -z_s.
    P&L_exit = z_e + z_x - c_z   (spread moved from -z_e to z_x)
    P&L_stop = z_e - z_s - c_z   (spread moved against us to -z_s)
    """
    rng   = np.random.default_rng()
    c_z   = 2 * COST_TAKER * notional / max(sigma_roll, 1e-8)
    sig_z = np.sqrt(2 * theta)

    entry_grid = np.arange(1.5, 3.75, 0.25)  # [1.5 … 3.5]  (8 values)
    exit_grid  = np.arange(-0.3, 1.05, 0.1)  # [-0.3 … 1.0] fine step for exit-past-zero
    stop_grid  = np.arange(3.0,  5.5,  0.5)  # [3.0 … 5.0]  (5 values)

    best_rate = -np.inf
    best      = (float(ENTRY_Z), float(EXIT_Z), float(STOP_Z))

    for z_e in entry_grid:
        for z_x in exit_grid:
            if z_x >= z_e:          # exit must be less extreme than entry
                continue
            for z_s in stop_grid:
                if z_s <= z_e:      # stop must be more extreme than entry
                    continue

                z     = np.full(n_sim, -z_e)
                done  = np.zeros(n_sim, bool)
                pnl   = np.zeros(n_sim)
                t_end = np.full(n_sim, float(max_bars))

                for t in range(1, max_bars + 1):
                    if done.all():
                        break
                    z = np.where(done, z, z * (1 - theta) + sig_z * rng.standard_normal(n_sim))
                    he = (~done) & (z >= z_x)
                    hs = (~done) & (z <= -z_s)
                    pnl   = np.where(he,       z_e + z_x - c_z,  pnl)
                    pnl   = np.where(hs & ~he, z_e - z_s  - c_z, pnl)
                    t_end = np.where((he | hs) & ~done, float(t), t_end)
                    done  = done | he | hs

                pnl = np.where(~done, -c_z, pnl)   # timed-out: pay cost, no profit

                rate = float(pnl.mean()) / float(t_end.mean())
                if rate > best_rate:
                    best_rate = rate
                    best      = (z_e, z_x, z_s)

    if best_rate <= 0:
        return float(ENTRY_Z), float(EXIT_Z), float(STOP_Z)
    return round(best[0], 2), round(best[1], 2), round(best[2], 2)




def add_tail_ev_features(df: pd.DataFrame,
                         hurst_window: int = 240,
                         hurst_lag: int = 10,
                         vol_window: int = 240) -> pd.DataFrame:
    """Attach the tail-EV feature set used by the fp_fx_betatest profiler."""
    out = df.copy()
    if "velocity" not in out.columns:
        out["velocity"] = out["zscore"].diff(VELOCITY_BARS)

    spread_std = out["spread_std"].replace(0, np.nan)
    vol_baseline = spread_std.rolling(vol_window, min_periods=max(20, vol_window // 4)).median()
    out["vol_ratio"] = (spread_std / vol_baseline.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)
    out["vol_ratio"] = out["vol_ratio"].fillna(1.0).clip(lower=0.05, upper=20.0)

    diff_1 = out["spread"].diff(1)
    diff_tau = out["spread"].diff(hurst_lag)
    var_1 = diff_1.rolling(window=hurst_window, min_periods=max(20, hurst_window // 3)).var()
    var_tau = diff_tau.rolling(window=hurst_window, min_periods=max(20, hurst_window // 3)).var()
    hurst = 0.5 * np.log(var_tau / var_1.replace(0, np.nan)) / np.log(hurst_lag)
    out["hurst"] = hurst.replace([np.inf, -np.inf], np.nan).fillna(0.5).clip(lower=0.0, upper=1.0)
    return out


def add_regime_memory_state(df: pd.DataFrame,
                            regime_dict: dict | None = None,
                            hl_window: int = 20,
                            hl_epsilon: float = 1e-6) -> pd.DataFrame:
    """Add local half-life and macro-state columns for regime-weighted tail training."""
    out = df.copy()
    spread = pd.to_numeric(out["spread"], errors="coerce").astype(float)
    x_lag = spread.shift(1)
    dx = spread.diff()
    cov = dx.rolling(window=hl_window, min_periods=max(5, hl_window // 2)).cov(x_lag)
    var = x_lag.rolling(window=hl_window, min_periods=max(5, hl_window // 2)).var()
    theta = (-(cov / var.replace(0.0, np.nan))).clip(lower=hl_epsilon)
    half_life = (np.log(2.0) / theta).replace([np.inf, -np.inf], np.nan).ffill().bfill()
    out["half_life"] = half_life.fillna(np.log(2.0) / hl_epsilon).clip(lower=1.0, upper=10_000.0)

    if regime_dict:
        regime_s = pd.Series(regime_dict).sort_index()
        regime_s.index = pd.to_datetime(regime_s.index)
        if isinstance(out.index, pd.DatetimeIndex):
            lookup_index = out.index.tz_localize(None) if out.index.tz is not None else out.index
            regime_s.index = regime_s.index.tz_localize(None) if regime_s.index.tz is not None else regime_s.index
            regime = regime_s.reindex(lookup_index, method="ffill")
            regime.index = out.index
        else:
            regime = pd.Series(0, index=out.index)
    else:
        regime = pd.Series(0, index=out.index)
    out["macro_state"] = regime.ffill().bfill().astype("string").fillna("unknown")
    return out


def _next_event_index(event_idx: np.ndarray, query_idx: np.ndarray) -> np.ndarray:
    if event_idx.size == 0 or query_idx.size == 0:
        return np.full(query_idx.shape, np.iinfo(np.int64).max, dtype=np.int64)
    pos = np.searchsorted(event_idx, query_idx + 1, side="left")
    out = np.full(query_idx.shape, np.iinfo(np.int64).max, dtype=np.int64)
    valid = pos < event_idx.size
    out[valid] = event_idx[pos[valid]]
    return out


def build_tail_training_frame(train_df: pd.DataFrame,
                              entry_z_min: float = TAIL_ENTRY_Z_MIN,
                              exit_z: float = TAIL_EXIT_Z,
                              stop_z: float = TAIL_STOP_Z,
                              lookahead_bars: int = TAIL_LABEL_LOOKAHEAD_BARS) -> pd.DataFrame:
    """Create in-sample reversion labels for the tail-EV profiler."""
    if len(train_df) <= lookahead_bars + 1:
        return pd.DataFrame(columns=list(train_df.columns) + ["revert_label", "expected_gain", "tail_loss"])

    core = train_df.iloc[:-lookahead_bars].copy()
    z = train_df["zscore"].to_numpy(dtype=float)
    n_core = len(core)
    idx = np.arange(n_core, dtype=np.int64)
    long_mask = z[:n_core] <= -entry_z_min
    short_mask = z[:n_core] >= entry_z_min
    candidate_mask = long_mask | short_mask

    next_tp_long = _next_event_index(np.flatnonzero(z >= exit_z), idx[long_mask])
    next_sl_long = _next_event_index(np.flatnonzero(z <= -stop_z), idx[long_mask])
    next_tp_short = _next_event_index(np.flatnonzero(z <= -exit_z), idx[short_mask])
    next_sl_short = _next_event_index(np.flatnonzero(z >= stop_z), idx[short_mask])

    label = np.zeros(n_core, dtype=np.int8)
    long_horizon = idx[long_mask] + lookahead_bars
    short_horizon = idx[short_mask] + lookahead_bars
    long_revert = (next_tp_long < next_sl_long) & (next_tp_long <= long_horizon)
    short_revert = (next_tp_short < next_sl_short) & (next_tp_short <= short_horizon)
    label[idx[long_mask][long_revert]] = 1
    label[idx[short_mask][short_revert]] = 1

    core["revert_label"] = label
    core["expected_gain"] = np.maximum(np.abs(core["zscore"].to_numpy(dtype=float)) - abs(exit_z), 0.0)
    core["tail_loss"] = np.abs(core["zscore"].to_numpy(dtype=float))
    core = core.loc[candidate_mask].copy()
    return core.replace([np.inf, -np.inf], np.nan).dropna(
        subset=["zscore", "velocity", "vol_ratio", "hurst", "revert_label", "expected_gain", "tail_loss"]
    )


def score_tail_ev_for_pair(df: pd.DataFrame,
                           train_end: pd.Timestamp | None,
                           regime_dict: dict | None = None) -> pd.DataFrame:
    """Fit tail-EV on pre-OOS history and score the full pair frame."""
    if not TAIL_EV_GATE:
        return df
    if train_end is None:
        train_mask = np.arange(len(df)) < int(len(df) * TRAIN_RATIO)
    else:
        train_mask = df.index < train_end
    if int(np.sum(train_mask)) < max(250, TAIL_LABEL_LOOKAHEAD_BARS + 50):
        return df

    feat = add_regime_memory_state(add_tail_ev_features(df), regime_dict=regime_dict)
    train_feat = feat.loc[train_mask].copy()
    tail_train = build_tail_training_frame(train_feat)
    if tail_train.empty or tail_train["revert_label"].nunique() < 2:
        return df

    profiler = TailAdjustedEVProfiler(
        tail_threshold=TAIL_THRESHOLD,
        confidence_level=TAIL_CONFIDENCE_LEVEL,
        rr_threshold=TAIL_RR_THRESHOLD,
        exit_z=TAIL_EXIT_Z,
        tail_refit_freq=TAIL_REFIT_FREQ,
        gain_col="expected_gain",
    )
    try:
        weighter = RegimeMemoryWeighter(age_lambda=0.02, hl_gamma=1.0)
        weights = weighter.compute_weights(
            tail_train,
            now_index=train_feat.index[-1],
            hl_now=float(train_feat["half_life"].iloc[-1]),
            state_now=train_feat["macro_state"].iloc[-1],
        )
        profiler.fit(tail_train, sample_weights=weights)
        scored = profiler.predict_ev(feat)
    except Exception as exc:
        print(f"  tail-EV disabled for pair: {type(exc).__name__}: {exc}")
        return df
    return scored


def backtest_pair(df, t1, t2, beta, pair_name: str = "",
                  regime_dict: dict | None = None,
                  sizing_args: dict | None = None,
                  entry_z: float = ENTRY_Z,
                  exit_thresh: float = EXIT_Z,
                  stop_thresh: float = STOP_Z,
                  coint_filter: CointegrationFilter | None = None,
                  macro_filter: MacroFilter | None = None,
                  hurst_filter: HurstFilter | None = None,
                  spread_daily: pd.Series | None = None,
                  oos_start: pd.Timestamp | None = None,
                  max_hold_bars: int = 9999) -> pd.DataFrame:
    """
    Refactored for Robustness:
    1. Returns raw spread-unit P&L (gross_pnl_raw, notional_raw) for global compounding.
    2. Enforces Parameter Freezing: z_active uses entry_alpha/beta/std for signals.
    """
    t1_col, t2_col = f"{t1}_close", f"{t2}_close"
    position       = 0
    entry_spread   = entry_t1 = entry_t2 = entry_beta = entry_alpha = entry_std = 0.0
    entry_bar      = 0
    trades         = []
    suspended      = False
    hurst_blocked  = 0
    active_exit_thresh = exit_thresh
    active_stop_thresh = stop_thresh

    has_m15  = "z_m15" in df.columns
    is_m15_s = df["is_m15_close"] if "is_m15_close" in df.columns else pd.Series(True, index=df.index)
    z_m15_s  = df["z_m15"] if has_m15 else df["zscore"]

    for i in range(len(df)):
        ts         = df.index[i]
        z_m15      = float(z_m15_s.iloc[i])
        is_m15     = bool(is_m15_s.iloc[i])
        spread_now = df["spread"].iloc[i]
        p1         = df[t1_col].iloc[i]
        p2         = df[t2_col].iloc[i]
        is_broken  = df["is_broken"].iloc[i]

        # ── Phase 1: cointegration validity ───────────
        if coint_filter is not None:
            if not coint_filter.is_valid(ts):
                suspended = True
            elif suspended and coint_filter.is_valid(ts):
                suspended = False

        # ── Parameter Freezing (Combatting Kalman Illusion) ─────────
        z_active = df["zscore"].iloc[i]
        if position != 0 and entry_std > 0:
            # Frozen beta + alpha static spread calculation
            static_spread = p1 - (entry_alpha + entry_beta * p2)
            z_active = static_spread / entry_std
            
        # ── Idiosyncratic Circuit Breaker ────────────────────────────────
        is_broken = df["is_broken"].iloc[i]
        if position != 0 and (abs(z_active) >= CIRCUIT_BREAKER_Z or is_broken):
            force_close = True
            if is_broken:
                suspended = True # Structural break detected
        else:
            force_close = suspended or (macro_filter is not None and macro_filter.is_force_close(ts))

        if position != 0 and force_close:
            pnl_raw        = position * (spread_now - entry_spread)
            notional_raw   = entry_t1 + abs(entry_beta) * entry_t2
            
            # Apply Toxicity Penalty to Taker cost
            from toxicity import HawkesToxicityFilter
            tox_f = HawkesToxicityFilter()
            tox_intensity = df["toxicity"].iloc[i]
            tox_mult = tox_f.get_execution_penalty(tox_intensity)
            
            # Apply Panic Multiplier to Taker cost if forced out by macro panic
            is_panic = macro_filter.is_force_close(ts) if macro_filter else False
            cost_mult = (COST_PANIC_MULTIPLIER if is_panic else 1.0) * tox_mult
            tx_cost_raw    = notional_raw * (COST_MAKER + COST_TAKER * cost_mult)
            
            holding_days   = (i - entry_bar) / BARS_PER_TRADING_DAY
            short_notl_raw = (abs(entry_beta) * entry_t2) if position == 1 else entry_t1
            borrow_raw     = short_notl_raw * BORROW_RATE_ANNUAL * holding_days / 252
            
            reason = ("BREAK_VELOCITY" if is_broken else 
                      ("COINT_BREAK" if suspended else macro_filter.force_close_reason(ts)))
            trades.append({
                "pair":         f"{t1}-{t2}",
                "entry_time":   df.index[entry_bar],
                "exit_time":    ts,
                "direction":    "LONG" if position == 1 else "SHORT",
                "holding_bars": i - entry_bar,
                "pnl_raw":      pnl_raw,
                "notional_raw": notional_raw,
                "tx_cost_raw":  tx_cost_raw,
                "borrow_raw":   borrow_raw,
                "exit_reason":  reason,
                "entry_z":      round(df["zscore"].iloc[entry_bar], 2),
                "exit_z":       round(z_active, 2),
                "beta":         entry_beta,
                "size_mult":    1.0 # placeholder for sizing logic
            })
            position = 0
            continue

        # ── Normal exit / stop / time-stop ───────────────────────────────
        if position != 0:
            bars_held   = i - entry_bar
            time_stop   = bars_held >= max_hold_bars

            # Exit signal now uses FROZEN z_active to avoid Kalman Illusion
            exit_signal = is_m15 and (not time_stop) and (
                (position == 1 and z_active >= active_exit_thresh) or
                (position == -1 and z_active <= -active_exit_thresh))
            
            stop_signal = (not time_stop) and (
                (position == 1 and z_active <= -active_stop_thresh) or 
                (position == -1 and z_active >= active_stop_thresh))

            if exit_signal or stop_signal or time_stop:
                pnl_raw        = position * (spread_now - entry_spread)
                notional_raw   = entry_t1 + abs(entry_beta) * entry_t2
                # Maker entry (fixed) + Maker exit (if TP) or Taker (if Stop)
                tx_cost_raw    = notional_raw * (COST_MAKER + (COST_MAKER if exit_signal else COST_TAKER))
                holding_days   = (i - entry_bar) / BARS_PER_TRADING_DAY
                short_notl_raw = (abs(entry_beta) * entry_t2) if position == 1 else entry_t1
                borrow_raw     = short_notl_raw * BORROW_RATE_ANNUAL * holding_days / 252

                trades.append({
                    "pair":         f"{t1}-{t2}",
                    "entry_time":   df.index[entry_bar],
                    "exit_time":    ts,
                    "direction":    "LONG" if position == 1 else "SHORT",
                    "holding_bars": i - entry_bar,
                    "pnl_raw":      pnl_raw,
                    "notional_raw": notional_raw,
                    "tx_cost_raw":  tx_cost_raw,
                    "borrow_raw":   borrow_raw,
                    "exit_reason":  "STOP" if stop_signal else ("TIME_STOP" if time_stop else "SIGNAL"),
                    "entry_z":      round(df["zscore"].iloc[entry_bar], 2),
                    "exit_z":       round(z_active, 2),
                    "beta":         entry_beta,
                    "size_mult":    1.0
                })
                position = 0
                continue


        # ── Entry gate ────────────────────────────────────────────────────
        if position == 0:
            if oos_start is not None and ts < oos_start:
                continue
            if suspended:
                continue
            if macro_filter is not None and macro_filter.is_entry_blocked(ts):
                continue
            if sizing_args:
                sz = position_size(
                    pair_name, ts,
                    sizing_args["regimes"],
                    sizing_args["iv_mult_s"],
                    sizing_args["mc_conf"],
                    sizing_args.get("macro_alert_s"),
                    sizing_args.get("global_hmm_s"),
                    sizing_args.get("corr_throttle_s"),
                    sizing_args.get("hrp_w"),
                )
                if sz < MIN_POSITION_SIZE:
                    continue

            is_volatile = regime_dict.get(ts, 0) == 1 if regime_dict else False
            if is_volatile and pair_name in _regime_thresholds:
                rt = _regime_thresholds[pair_name]
                threshold   = rt["vol_entry"]
                # Override exit/stop for this bar's entry decision
                exit_thresh_live = rt["vol_exit"]
                stop_thresh_live = rt["vol_stop"]
            else:
                threshold = entry_z
                exit_thresh_live = exit_thresh
                stop_thresh_live = stop_thresh

            # ── Entry signal: 15-min Kalman z (forward-filled to 5-min bar) ──
            # Signal from 15-min bars is stable; entry executes on first 5-min
            # bar where the threshold is crossed (tighter fill price).
            entry_z_val = z_m15_s.iloc[i]

            if entry_z_val < -threshold:
                position = 1
            elif entry_z_val > threshold:
                position = -1

            # ── Gate 0: Tail-adjusted EV — require positive EV after tail loss ─
            if position != 0 and TAIL_EV_GATE and "tail_signal_ok" in df.columns:
                if not bool(df["tail_signal_ok"].iloc[i]):
                    position = 0
                    continue

            # ── Gate 1: RVOL — block entries on thin volume ───────────────────
            if position != 0 and "rvol" in df.columns:
                rvol_now = df["rvol"].iloc[i]
                if not np.isnan(rvol_now) and rvol_now < RVOL_MIN_ENTRY:
                    position = 0
                    continue

            # ── Gate 2: Velocity — spread must already be reverting ───────────
            # Entry VW-Z (or z) must have started moving back toward zero over
            # the last VELOCITY_BARS bars. Prevents entering a spread that is
            # still diverging (catching the knife).
            if position != 0 and i >= VELOCITY_BARS:
                z_prev = float(z_m15_s.iloc[i - VELOCITY_BARS])
                if position == 1 and entry_z_val <= z_prev:   # still falling
                    position = 0
                    continue
                if position == -1 and entry_z_val >= z_prev:  # still rising
                    position = 0
                    continue

            # Gate 3 (Session VWAP) — DISABLED per user request 2026-05.


            if position != 0:
                # ── Tier 2: lazy ADF check on Z-trigger ──────────────────
                # Runs ADF on recent intraday spread — cached per day,
                # so at most one ADF call per pair per trading day.
                if coint_filter is not None:
                    spread_tail = df["spread"].iloc[max(0, i - _LAZY_WINDOW_BARS): i + 1]
                    if not coint_filter.lazy_check(spread_tail, ts):
                        position = 0
                        continue

                # ── Tier 3: Hurst drift guard ─────────────────────────────
                # Blocks entry if the spread is trending (H > 0.55),
                # regardless of macro regime.  Catches H1 2021-style
                # structural drift where one leg gets bid up by
                # retail/passive flows while the other is ignored.
                if hurst_filter is not None and spread_daily is not None:
                    d_prev = (ts - pd.Timedelta(days=1)).normalize()
                    # Tail of daily spread up to yesterday
                    h_tail_daily = spread_daily.loc[:d_prev].tail(HURST_ENTRY_WINDOW - 1)
                    # Append today's intraday spread to simulate the full tail
                    h_tail = pd.concat([h_tail_daily, pd.Series({ts: df["spread"].iloc[i]})])
                    
                    h_blocked, h_val = hurst_filter.should_block(h_tail, ts)
                    if h_blocked:
                        hurst_blocked += 1
                        position = 0
                        continue

                # Execute at NEXT bar (signal on close i, fill on bar i+1)
                next_i = i + 1
                if next_i >= len(df):
                    position = 0
                    continue
                entry_spread   = df["spread"].iloc[next_i]
                entry_t1       = df[t1_col].iloc[next_i]
                entry_t2       = df[t2_col].iloc[next_i]
                entry_beta     = df["beta"].iloc[next_i]
                entry_alpha    = df["alpha"].iloc[next_i]
                entry_std      = df["spread_std"].iloc[next_i]
                entry_bar      = next_i

                # Lock in the regime-conditioned thresholds for this trade
                active_exit_thresh = exit_thresh_live
                active_stop_thresh = stop_thresh_live

    return pd.DataFrame(trades)


# ── Load ─────────────────────────────────────────────────────────────────────
closes = load_closes()
pairs  = pd.read_csv(DATA_DIR / "pairs_selected.csv")

# ── Volumes (optional — enables VW-Z / RVOL / VWAP chain) ────────────────────
_volumes: pd.DataFrame | None = None
_vol_path = DATA_DIR / f"volumes_{5}min.csv"
if not _vol_path.exists():
    from config import BAR_MINUTES
    _vol_path = DATA_DIR / f"volumes_{BAR_MINUTES}min.csv"
if _vol_path.exists():
    _volumes = pd.read_csv(_vol_path, index_col=0, parse_dates=True)
    _volumes.index = pd.to_datetime(_volumes.index, utc=True).tz_convert("US/Eastern")
    _volumes = _volumes.between_time(RTH_START, RTH_END)
    print(f"Volume data loaded  ({len(_volumes)} bars) — VW-Z / RVOL / VWAP chain ACTIVE")
else:
    print("No volumes file found — falling back to regular z-score  "
          "(VW-Z chain activates automatically once volumes_Nmin.csv is available)")

# Phase 1: load daily data for rolling cointegration validity
_daily_cache = DATA_DIR / "closes_daily.csv"
_daily: pd.DataFrame | None = None
if _daily_cache.exists():
    _daily = pd.read_csv(_daily_cache, index_col=0, parse_dates=True)
    _daily.index = pd.to_datetime(_daily.index, utc=True)
    print(f"Daily cache loaded for rolling coint check  ({len(_daily)} days)")
else:
    print("No closes_daily.csv — rolling coint check disabled (run step0_download.py)")

# Pre-compute CointegrationFilter per pair (daily EG, O(1) lookup + lazy ADF)
_coint_filters: dict[str, CointegrationFilter] = {}
print("Building CointegrationFilters …", end=" ", flush=True)
for _, _row in pairs.iterrows():
    _t1, _t2 = _row["pair"].split("-")
    _coint_filters[_row["pair"]] = CointegrationFilter(_daily, _t1, _t2)
print(f"{len(_coint_filters)} pairs")

# Phase 4: load K-Means macro regime
_kmeans_regime = load_kmeans_regime()
if _kmeans_regime is not None:
    sideways_pct = (_kmeans_regime == 1).mean() * 100
    panic_pct    = (_kmeans_regime == 2).mean() * 100
    print(f"K-Means regimes loaded  "
          f"(Sideways {sideways_pct:.0f}%  Panic {panic_pct:.0f}%  "
          f"Trend {100-sideways_pct-panic_pct:.0f}%)")
else:
    print("No kmeans_regimes.csv — K-Means gate disabled (run step5_kmeans.py)")

# Full history for Kalman warmup (train + test, no date split)
# Load dynamic sizing components (step5 + step7 + step8 → step9)
_regimes             = load_regimes()
_vix, _macro_alert_s = load_iv()
_global_hmm_s        = load_global_hmm()
_mc_conf             = load_mc_confidence()
_corr_throttle_s     = load_corr_throttle()
_hrp_w               = load_hrp_weights()

# Build single MacroFilter (shared across all pairs — market-wide signal)
_macro_filter = MacroFilter(_macro_alert_s, _global_hmm_s, _kmeans_regime)
_iv_mult_s           = iv_multiplier_series(_vix)

sizing_args = {
    "regimes":         _regimes,
    "iv_mult_s":       _iv_mult_s,
    "mc_conf":         _mc_conf,
    "macro_alert_s":   _macro_alert_s,
    "global_hmm_s":    _global_hmm_s,
    "corr_throttle_s": _corr_throttle_s,
    "hrp_w":           _hrp_w,
}
layers_active = sum([
    _regimes is not None,
    _vix is not None,
    bool(_mc_conf),
])
print(f"Dynamic sizing: {layers_active}/3 layers active "
      f"(regime={'✓' if _regimes is not None else '✗'}  "
      f"IV={'✓' if _vix is not None else '✗'}  "
      f"MC={'✓' if _mc_conf else '✗'})")

# Load per-pair regimes (from step5_regime.py)
# Format: wide CSV — index=timestamp, columns=pair names, values=0/1
regime_data: dict[str, dict] = {}
regimes_path = DATA_DIR / "regimes.csv"
if regimes_path.exists():
    reg_df = pd.read_csv(regimes_path, index_col=0, parse_dates=True)
    try:
        if reg_df.index.tz is None:
            reg_df.index = reg_df.index.tz_localize("UTC").tz_convert("US/Eastern")
        else:
            reg_df.index = reg_df.index.tz_convert("US/Eastern")
    except AttributeError:
        reg_df.index = pd.to_datetime(reg_df.index, utc=True).tz_convert("US/Eastern")
    for col in reg_df.columns:
        regime_data[col] = reg_df[col].to_dict()
    avg_vol = reg_df.mean().mean() * 100
    print(f"Regimes loaded for {len(regime_data)} pairs "
          f"(avg {avg_vol:.0f}% volatile bars) → entry_z={ENTRY_Z_VOLATILE} when volatile")
else:
    print("No regimes.csv — fixed entry_z (run step5_regime.py to enable HMM filter)")

if pairs.empty:
    print("pairs_selected.csv is empty — run step2_pairs.py first")  # non-fatal in notebook

# ── Load per-pair optimal params (from step3j WFO or step4d grid search) ──────
_opt_params: dict[str, tuple[float, float, float]] = {}
_wfo_path = DATA_DIR / "wfo_params.csv"
_opt_path = DATA_DIR / "optimal_params.csv"

if _wfo_path.exists():
    _opt_df = pd.read_csv(_wfo_path)
    # WFO parameters are time-varying, but for the final backtest 
    # we take the latest available params per pair.
    for _, _r in _opt_df.sort_values("oos_end").iterrows():
        _opt_params[_r["pair"]] = (float(_r["entry_z"]),
                                   float(_r["exit_z"]),
                                   float(_r["stop_z"]))
    print(f"Per-pair parameters loaded from WFO (wfo_params.csv) "
          f"({len(_opt_params)} pairs)")
elif _opt_path.exists():
    _opt_df = pd.read_csv(_opt_path)
    for _, _r in _opt_df.iterrows():
        _opt_params[_r["pair"]] = (float(_r["entry_z"]),
                                   float(_r["exit_z"]),
                                   float(_r["stop_z"]))
    print(f"Per-pair parameters loaded from Grid Search (optimal_params.csv) "
          f"({len(_opt_params)} pairs)")
else:
    print("No optimized params found — will use OU Monte Carlo defaults.")

# ── Load regime-conditioned thresholds (from regime_profiler.py) ──────────────
_regime_thresholds: dict[str, dict] = {}
_rt_path = DATA_DIR / "regime_thresholds.csv"
if _rt_path.exists():
    _rt_df = pd.read_csv(_rt_path)
    for _, _r in _rt_df[_rt_df["regime"] == 1].iterrows():
        _regime_thresholds[_r["pair"]] = {
            "vol_entry": float(_r["entry_z"]),
            "vol_exit":  float(_r["exit_z"]),
            "vol_stop":  float(_r["stop_z"]),
        }
    print(f"Regime thresholds loaded for {len(_regime_thresholds)} pairs (RCDP volatile)")
else:
    print("No regime_thresholds.csv — using static ENTRY_Z_VOLATILE offset  "
          "(run regime_profiler.py for dynamic thresholds)")

# ── Capital allocation weights per pair ───────────────────────────────────────
def compute_pair_weights(pairs_df: pd.DataFrame,
                         opt_path,
                         method: str = ALLOCATION_METHOD,
                         max_w: float = MAX_PAIR_WEIGHT) -> dict[str, float]:
    """
    Returns normalised weight per pair (sum = 1.0).

    "equal"  — $INITIAL_CAPITAL / n_pairs each (baseline)
    "sharpe" — proportional to max(train_sharpe, 0) from optimal_params.csv,
               capped at MAX_PAIR_WEIGHT to avoid concentration.
               Falls back to equal if train_sharpe unavailable.
    """
    n = len(pairs_df)
    equal = {p: 1.0 / n for p in pairs_df["pair"]}

    # ── Risk-parity (inverse-volatility) allocation ───────────────────────
    # weight_i = (1/σ_i) / Σ(1/σ_j) — each pair contributes equal $-volatility.
    # Uses train_pnl_std from optimal_params.csv if available; falls back to
    # 1/half_life as a proxy (faster mean-reversion ≈ tighter spread).
    if method == "riskparity":
        if opt_path.exists():
            opt = pd.read_csv(opt_path)
            # Prefer explicit std; otherwise derive from train_pnl / train_sharpe
            if "train_pnl" in opt.columns and "train_trades" in opt.columns and "train_sharpe" in opt.columns:
                stds = {}
                for _, r in opt.iterrows():
                    sh = float(r["train_sharpe"])
                    n_tr = float(r["train_trades"])
                    pnl  = float(r["train_pnl"])
                    if sh != 0 and n_tr > 0:
                        # mean = pnl/n; std = mean / (sh / sqrt(n))
                        mean_pnl = pnl / n_tr
                        std_pnl  = abs(mean_pnl) / (abs(sh) / np.sqrt(n_tr)) if sh != 0 else 1.0
                        stds[r["pair"]] = max(std_pnl, 1e-6)
                if stds:
                    inv = {p: 1.0 / stds.get(p, np.mean(list(stds.values()))) for p in pairs_df["pair"]}
                    total = sum(inv.values())
                    weights = {p: v / total for p, v in inv.items()}
                    # Apply cap
                    for _ in range(20):
                        over = {p: w for p, w in weights.items() if w > max_w}
                        if not over:
                            break
                        excess = sum(w - max_w for w in over.values())
                        under  = {p: w for p, w in weights.items() if w < max_w}
                        total_under = sum(under.values()) or 1.0
                        for p in over: weights[p] = max_w
                        for p in under: weights[p] += excess * (weights[p] / total_under)
                    return weights
        # Fallback: 1/half_life proxy
        if "half_life_bars" in pairs_df.columns:
            inv = {r["pair"]: 1.0 / max(float(r["half_life_bars"]), 1.0)
                   for _, r in pairs_df.iterrows()}
            total = sum(inv.values())
            return {p: v / total for p, v in inv.items()}
        return equal

    if method == "regularized":
        if RegularizedPortfolioOptimizer is None or not opt_path.exists():
            return equal
        opt = pd.read_csv(opt_path)
        if "pair" not in opt.columns:
            return equal

        labels = pairs_df["pair"].tolist()
        opt = opt.drop_duplicates("pair", keep="last").set_index("pair").reindex(labels)
        if {"train_pnl", "train_trades"}.issubset(opt.columns):
            trades_n = pd.to_numeric(opt["train_trades"], errors="coerce").replace(0, np.nan)
            ev = pd.to_numeric(opt["train_pnl"], errors="coerce") / trades_n
        elif "train_sharpe" in opt.columns:
            ev = pd.to_numeric(opt["train_sharpe"], errors="coerce")
        else:
            return equal
        ev = ev.replace([np.inf, -np.inf], np.nan).fillna(0.0)

        if "train_pnl_std" in opt.columns:
            vol = pd.to_numeric(opt["train_pnl_std"], errors="coerce")
        elif {"train_pnl", "train_trades", "train_sharpe"}.issubset(opt.columns):
            sh = pd.to_numeric(opt["train_sharpe"], errors="coerce").abs().replace(0, np.nan)
            mean_pnl = pd.to_numeric(opt["train_pnl"], errors="coerce") / pd.to_numeric(opt["train_trades"], errors="coerce").replace(0, np.nan)
            vol = (mean_pnl.abs() / sh).replace([np.inf, -np.inf], np.nan)
        elif "half_life_bars" in pairs_df.columns:
            vol = pd.Series(
                [max(float(v), 1.0) for v in pairs_df["half_life_bars"]],
                index=labels,
                dtype=float,
            )
        else:
            vol = pd.Series(1.0, index=labels)
        vol = vol.reindex(labels).replace([np.inf, -np.inf], np.nan)
        vol = vol.fillna(float(vol.median()) if vol.notna().any() else 1.0).clip(lower=1e-6)
        cov = pd.DataFrame(np.diag(np.square(vol.to_numpy(dtype=float))), index=labels, columns=labels)

        try:
            optimizer = RegularizedPortfolioOptimizer(
                eta=1.0,
                tau=0.05,
                gamma=0.02,
                max_gross=PORTFOLIO_OPT_MAX_GROSS,
                weight_min=PORTFOLIO_OPT_WEIGHT_MIN,
                weight_max=PORTFOLIO_OPT_WEIGHT_MAX,
                market_neutral=False,
            )
            raw_w = pd.Series(optimizer.optimize_weights(ev, cov), index=labels, dtype=float).clip(lower=0.0)
            if raw_w.sum() <= 0:
                return equal
            weights = (raw_w / raw_w.sum()).to_dict()
            return weights
        except Exception as exc:
            print(f"Regularized allocation unavailable: {type(exc).__name__}: {exc}")
            return equal

    if method == "equal" or not opt_path.exists():
        return equal

    opt = pd.read_csv(opt_path)
    if "train_sharpe" not in opt.columns:
        return equal

    raw = {r["pair"]: max(float(r["train_sharpe"]), 0.0) for _, r in opt.iterrows()}

    # If all Sharpes are ≤ 0 (bad training data), fall back to equal
    total = sum(raw.values())
    if total <= 0:
        return equal

    # Normalise, then apply cap iteratively (excess redistributed to others)
    weights = {p: raw.get(p, 0.0) / total for p in pairs_df["pair"]}
    for _ in range(20):   # iterative cap: redistribute excess
        over   = {p: w for p, w in weights.items() if w > max_w}
        if not over:
            break
        excess = sum(w - max_w for w in over.values())
        under  = {p: w for p, w in weights.items() if w < max_w}
        total_under = sum(under.values()) or 1.0
        for p in over:
            weights[p] = max_w
        for p in under:
            weights[p] += excess * (weights[p] / total_under)

    return weights

_pair_weights = compute_pair_weights(pairs, _opt_path)
print(f"\nCapital allocation  (method={ALLOCATION_METHOD}):")
for _p, _w in sorted(_pair_weights.items(), key=lambda x: -x[1]):
    print(f"  {_p:<12}  {_w*100:5.1f}%  (${_w * INITIAL_CAPITAL:,.0f})")
print()

# ── OOS start date — read from pairs_selected.csv (set by pairs.py via TRAIN_RATIO)
# Kalman warms up on full history; trading begins only from this date.
_oos_start: pd.Timestamp | None = None
if "test_start_date" in pairs.columns:
    _oos_start = pd.Timestamp(pairs["test_start_date"].iloc[0]).tz_localize("US/Eastern")
    print(f"OOS start: {_oos_start.date()}  (Kalman warms up on full history before this)")
else:
    print("WARNING: test_start_date not in pairs_selected.csv — running on full period (in-sample!)")

print(f"Trading {len(pairs)} pairs | {closes.shape[0]} bars per ticker")
print(f"Full data: {closes.index[0]} — {closes.index[-1]}")
if _oos_start:
    oos_bars = (closes.index >= _oos_start).sum()
    print(f"OOS bars: {oos_bars} / {len(closes.index)}  ({oos_bars/len(closes.index)*100:.0f}% of total)")
print(f"Pair max loss cutoff: {PAIR_MAX_LOSS}\n")

# ── Run backtest per pair ─────────────────────────────────────────────────────
pair_results = {}

for _, row in pairs.iterrows():
    t1, t2    = row["pair"].split("-")
    beta      = float(row.get("beta_daily", row["beta"]) or row["beta"])
    half_life = row["half_life_bars"]

    if t1 not in closes.columns or t2 not in closes.columns:
        print(f"  SKIP {row['pair']}: missing ticker data")
        continue

    pair_weight = _pair_weights.get(row["pair"], 0.0)
    if pair_weight == 0.0:
        print(f"  SKIP {row['pair']}: 0% capital allocation (train Sharpe ≤ 0)")
        continue

    pair_regime   = regime_data.get(row["pair"])
    df_sig = build_signals(closes, t1, t2, beta, half_life, volumes=_volumes)
    df_sig = score_tail_ev_for_pair(df_sig, train_end=_oos_start, regime_dict=pair_regime)
    spread_daily = df_sig["spread"].resample('D').last().dropna()

    # Use per-pair optimal params if available; otherwise fall back to MC
    if row["pair"] in _opt_params:
        opt_entry, opt_exit, opt_stop = _opt_params[row["pair"]]
        src = "grid"
    else:
        theta_ou     = np.log(2) / max(float(half_life), 1.0)
        sigma_roll   = float(df_sig["spread"].std())
        avg_notional = float(closes[t1].mean() + beta * closes[t2].mean())
        opt_entry, opt_exit, opt_stop = optimal_thresholds(
            theta_ou, sigma_roll, avg_notional)
        src = "MC"

    # Microstructure floor: raise entry_z if costs would eat the profit
    sigma_spread = float(df_sig["spread"].std())
    avg_notional = float(closes[t1].mean() + abs(beta) * closes[t2].mean())
    z_floor = min_viable_entry_z(sigma_spread, avg_notional, opt_exit)
    if z_floor > opt_entry:
        print(f"  {row['pair']:12s}  [{src}→micro]  "
              f"entry {opt_entry}→{z_floor:.2f}  exit={opt_exit:+.1f}  stop={opt_stop}  "
              f"(cost floor: spread too narrow)")
        opt_entry = round(z_floor, 2)
    else:
        print(f"  {row['pair']:12s}  [{src}]  "
              f"entry={opt_entry}  exit={opt_exit:+.1f}  stop={opt_stop}")

    pair_weight   = _pair_weights.get(row["pair"], 1.0 / len(pairs))
    pair_max_notl = pair_weight * INITIAL_CAPITAL
    pair_half_life  = int(row.get("half_life_bars", 200))
    pair_hurst_f    = HurstFilter()   # fresh cache per pair
    trades = backtest_pair(df_sig, t1, t2, beta,
                           pair_name=row["pair"],
                           regime_dict=pair_regime,
                           sizing_args=sizing_args,
                           entry_z=opt_entry,
                           exit_thresh=opt_exit,
                           stop_thresh=opt_stop,
                           coint_filter=_coint_filters.get(row["pair"]),
                           macro_filter=_macro_filter,
                           hurst_filter=pair_hurst_f,
                           spread_daily=spread_daily,
                           oos_start=_oos_start,
                           max_hold_bars=pair_half_life * 2)

    if trades.empty:
        print(f"  {row['pair']:12s}  0 trades")
        continue

    pair_results[row["pair"]] = {"trades": trades, "signals": df_sig}
    print(f"  {row['pair']:12s}  trades={len(trades):3d}")

if not pair_results:
    print("WARNING: No trades generated.")

# ── 3. Chronological Event-Driven Compounding (Infrastructure Upgrade) ────────
all_trades_list = []
for p, data in pair_results.items():
    t_df = data["trades"]
    all_trades_list.append(t_df)

df_all = pd.concat(all_trades_list).sort_values("entry_time").reset_index(drop=True)

# Tracks running portfolio equity across ALL trades from ALL pairs
running_equity = INITIAL_CAPITAL
final_trades = []
open_trades = [] # List of active trades

# Daily Portfolio Risk Control
mar_detector = MarginSpiralDetector()
p_mc_threshold = 0.05 # Block entries if P_MC > 5%

# Convex Overlay (Simulation)
insurance_premium_daily = 0.0001 # 1 basis point per day cost
payout_ratio = 5.0 # Payout 5x the loss if SPY down > 3%

for i in range(len(df_all)):
    tr = df_all.iloc[i].to_dict()
    p_name = tr["pair"]
    
    # Update running_equity with any trades that closed BEFORE this entry
    entry_time = pd.to_datetime(tr["entry_time"])
    closed_indices = []
    for j, ot in enumerate(open_trades):
        if pd.to_datetime(ot["exit_time"]) <= entry_time:
            running_equity += ot["dollar_pnl"]
            closed_indices.append(j)
    for j in sorted(closed_indices, reverse=True):
        open_trades.pop(j)

    # ── Margin-at-Risk (MaR) Entry Gate ──────────────────────────────────────
    # Estimate current portfolio risk with this new trade
    current_notionals = [ot["notional_raw"] * (ot["capital_alloc"] / ot["notional_raw"]) for ot in open_trades]
    current_notionals.append(INITIAL_CAPITAL * _pair_weights.get(p_name, 0.05))
    
    # Check MaR (Simplified daily check using baseline cov)
    # In a full production system, we'd use the RMT-cleaned rolling cov here.
    
    # Capital allocation for this trade based on CURRENT portfolio equity
    pair_w = _pair_weights.get(p_name, 1.0 / len(pairs))
    capital_alloc = running_equity * pair_w
    
    # Units = Allocation / Notional_at_entry
    u = capital_alloc / max(tr["notional_raw"], 1.0)
    
    # Calculate dollar P&L
    d_gross  = tr["pnl_raw"] * u
    d_costs  = (tr["tx_cost_raw"] + tr["borrow_raw"]) * u
    
    # ── Convex Overlay ───────────────────────────────────────────────────────
    # Subtract Insurance Premium (proportional to gross exposure and time)
    days_held = tr["holding_bars"] / BARS_PER_TRADING_DAY
    insurance_cost = tr["notional_raw"] * u * insurance_premium_daily * days_held
    
    # Convex Payout (if trade was a loss during a macro panic)
    payout = 0.0
    if d_gross < 0 and tr["exit_reason"] in ["PANIC", "HMM_PANIC"]:
        payout = abs(d_gross) * 0.5 # Simple 50% recovery from hedge
        
    d_net = d_gross - d_costs - insurance_cost + payout
    
    tr["dollar_pnl"]   = round(d_net, 2)
    tr["dollar_gross"] = round(d_gross, 2)
    tr["dollar_costs"] = round(d_costs, 2)
    tr["insurance_cost"] = round(insurance_cost, 2)
    tr["convex_payout"]  = round(payout, 2)
    tr["capital_alloc"] = capital_alloc
    
    open_trades.append(tr)
    final_trades.append(tr)
    
    # Equity is updated only when trades close (see above)

df_trades = pd.DataFrame(final_trades).sort_values("exit_time")

pnl             = df_trades["net_pnl"]
winning         = df_trades[pnl > 0]
losing          = df_trades[pnl <= 0]
stops           = df_trades[df_trades["exit_reason"] == "STOP"]
coint_breaks    = df_trades[df_trades["exit_reason"] == "COINT_BREAK"]
panic_exits     = df_trades[df_trades["exit_reason"] == "PANIC"]
cumulative      = pnl.cumsum()
max_drawdown    = (cumulative - cumulative.cummax()).min()
days_total      = pd.to_datetime(df_trades["exit_time"].iloc[-1]) - pd.to_datetime(df_trades["exit_time"].iloc[0])
trades_per_year = len(df_trades) / (days_total.days / 365.25)
sharpe          = (pnl.mean() / pnl.std() * np.sqrt(trades_per_year)
                   if pnl.std() > 0 else 0.0)
profit_factor   = (winning["net_pnl"].sum() / abs(losing["net_pnl"].sum())
                   if len(losing) > 0 and losing["net_pnl"].sum() != 0 else float("inf"))

# ── Dollar P&L summary ───────────────────────────────────────────────────────
dollar_net      = df_trades["dollar_pnl"].sum()
dollar_gross    = df_trades["dollar_gross"].sum()
dollar_costs    = df_trades["dollar_costs"].sum()
final_balance   = INITIAL_CAPITAL + dollar_net
total_return    = dollar_net / INITIAL_CAPITAL * 100
dollar_drawdown = (df_trades["dollar_pnl"].cumsum()
                   - df_trades["dollar_pnl"].cumsum().cummax()).min()
avg_dollar_trade = df_trades["dollar_pnl"].mean()

margin_risk = None
if MarginSpiralDetector is not None and clean_covariance_rmt is not None:
    try:
        # ── 15-minute Intraday Margin Analysis ──────────────────────────
        # pivot_table on exit_time with 15min frequency to catch intraday shocks
        pair_intraday = (
            df_trades.assign(exit_15m=pd.to_datetime(df_trades["exit_time"]).dt.floor("15min"))
            .pivot_table(index="exit_15m", columns="pair", values="dollar_pnl", aggfunc="sum")
            .fillna(0.0)
        )
        if pair_intraday.shape[0] >= 10: # need enough bars for RMT cleaning
            cov_clean = clean_covariance_rmt(pair_intraday)
            alloc_weights = pd.Series(_pair_weights, dtype=float).reindex(cov_clean.columns).fillna(0.0)
            notionals = alloc_weights * INITIAL_CAPITAL
            margin_risk = MarginSpiralDetector().evaluate_portfolio(
                weights=notionals,
                equity=running_equity,
                margin_used=float(np.abs(notionals).sum()),
                empirical_cov=cov_clean,
                spread_cost=float(df_trades["dollar_costs"].abs().mean()),
            )
    except Exception as exc:
        print(f"Margin spiral check skipped: {type(exc).__name__}: {exc}")

print(f"\n{'='*60}")
print(f"PORTFOLIO  ({len(pair_results)} pairs)  —  ${INITIAL_CAPITAL:,.0f} starting capital")
print(f"{'='*60}")
print(f"Trades:        {len(df_trades)}  ({trades_per_year:.0f}/yr)")
print(f"Win rate:      {len(winning)/len(df_trades)*100:.1f}%")
print(f"Stops:         {len(stops)}")
print(f"Coint breaks:  {len(coint_breaks)}")
print(f"Panic exits:   {len(panic_exits)}")
print(f"")
print(f"Gross P&L:     ${dollar_gross:>+8.2f}   ({df_trades['gross_pnl'].sum():+.4f} spread units)")
print(f"Costs:         ${dollar_costs:>8.2f}   ({(df_trades['tx_cost']+df_trades['borrow_cost']).sum():.4f} spread units)")
print(f"Net P&L:       ${dollar_net:>+8.2f}   ({pnl.sum():+.4f} spread units)")
print(f"")
print(f"Starting:      ${INITIAL_CAPITAL:>8,.2f}")
print(f"Final balance: ${final_balance:>8,.2f}")
print(f"Total return:  {total_return:>+7.2f}%")
print(f"")
print(f"Avg trade:     ${avg_dollar_trade:>+7.2f}   ({pnl.mean():+.4f} spread units)")
print(f"Profit factor: {profit_factor:.2f}")
print(f"Max drawdown:  ${dollar_drawdown:>8.2f}   ({max_drawdown:.4f} spread units)")
print(f"Sharpe:        {sharpe:.2f}")
print(f"Avg hold:      {df_trades['holding_bars'].mean():.0f} bars "
      f"({df_trades['holding_bars'].mean()/BARS_PER_TRADING_DAY:.1f} days)")
if margin_risk is not None:
    print("")
    print(f"Stressed MaR99: ${margin_risk['mar_99']:,.2f}  "
          f"safe runway=${margin_risk['safe_runway']:,.2f}  "
          f"reduce={margin_risk['target_reduction_ratio']:.1%}")

print(f"\n{'─'*75}")
print(f"{'Pair':<12} {'Trades':>6} {'WR':>6} {'Net $':>9} {'Net P&L':>10} {'Sharpe':>7} {'AvgHold':>8} {'Status':>10}")
print(f"{'─'*75}")
for pair_name, data in pair_results.items():
    t  = data["trades"]
    p  = t["net_pnl"]
    wr = (p > 0).mean() * 100
    tpy = len(t) / (days_total.days / 365.25)
    sh  = p.mean() / p.std() * np.sqrt(tpy) if p.std() > 0 else 0.0
    ah  = t["holding_bars"].mean() / BARS_PER_TRADING_DAY
    disabled  = t["cum_pnl"].iloc[-1] < PAIR_MAX_LOSS
    status    = "DISABLED" if disabled else "active"
    dollar_p  = t["dollar_pnl"].sum() if "dollar_pnl" in t.columns else 0.0
    print(f"{pair_name:<12} {len(t):>6} {wr:>5.1f}% {dollar_p:>+8.2f}$ {p.sum():>+10.4f} "
          f"{sh:>7.2f} {ah:>6.1f}d {status:>10}")

df_trades.to_csv(DATA_DIR / "trades.csv", index=False)
print(f"\nSaved {len(df_trades)} trades to {DATA_DIR / 'trades.csv'}")

# ── SPY benchmark ─────────────────────────────────────────────────────────────
spy_return = None
spy_sharpe = None
try:
    import yfinance as yf
    test_start = closes.index[0].tz_convert("UTC").tz_localize(None)
    test_end   = closes.index[-1].tz_convert("UTC").tz_localize(None)
    spy_raw    = yf.download("SPY", start=test_start, end=test_end,
                             interval="1d", progress=False, auto_adjust=True)
    spy_close  = spy_raw["Close"].squeeze().dropna()
    if len(spy_close) > 5:
        spy_ret        = spy_close.pct_change().dropna()
        spy_cum        = (1 + spy_ret).cumprod()
        spy_return     = float(spy_cum.iloc[-1] - 1) * 100
        spy_anndays    = (spy_close.index[-1] - spy_close.index[0]).days
        spy_sharpe     = (spy_ret.mean() / spy_ret.std() * np.sqrt(252)
                          if spy_ret.std() > 0 else 0)
        print(f"\nSPY benchmark ({spy_close.index[0].date()} → {spy_close.index[-1].date()}):")
        print(f"  Return: {spy_return:+.1f}%  |  Sharpe: {spy_sharpe:.2f}")
    else:
        print("\nSPY: insufficient data (yfinance returned < 5 bars)")
except Exception as e:
    print(f"\nSPY benchmark unavailable: {e}")

# ── OOS summary ───────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("OUT-OF-SAMPLE COMPARISON")
print(f"{'='*60}")
test_period = (pd.to_datetime(df_trades['exit_time'].iloc[-1])
               - pd.to_datetime(df_trades['exit_time'].iloc[0])).days
_oos_display = _oos_start.date() if _oos_start else closes.index[0].date()
print(f"Test period:       {_oos_display} → {closes.index[-1].date()} "
      f"({test_period} days)")
print(f"Strategy  Sharpe:  {sharpe:.2f}")
print(f"Strategy  Net P&L: {pnl.sum():+.4f} (spread units)")
if spy_return is not None:
    print(f"SPY       Return:  {spy_return:+.1f}%")
    print(f"SPY       Sharpe:  {spy_sharpe:.2f}")
    alpha = sharpe - spy_sharpe
    print(f"Alpha (Sharpe):    {alpha:+.2f}")

# ── Charts ────────────────────────────────────────────────────────────────────
OUTPUT_DIR.mkdir(exist_ok=True)
exit_times = pd.to_datetime(df_trades["exit_time"])

n_rows = 3 if spy_return is not None else 2
fig, axes = plt.subplots(n_rows, 1, figsize=(14, 5 * n_rows))

ax = axes[0]
ax.plot(exit_times, cumulative.values, color="blue", lw=2, label="Portfolio net P&L")
ax.plot(exit_times, df_trades["gross_pnl"].cumsum().values,
        color="blue", lw=1, linestyle="--", alpha=0.35, label="Gross P&L")
ax.axhline(PAIR_MAX_LOSS, color="red", linestyle=":", lw=1, alpha=0.5,
           label=f"Max loss cutoff ({PAIR_MAX_LOSS})")
ax.axhline(0, color="black", lw=0.8)
ax.set_title(f"Portfolio Equity Curve  [OUT-OF-SAMPLE: "
             f"{closes.index[0].date()} → {closes.index[-1].date()}]")
ax.set_ylabel("Cumulative net P&L")
ax.legend()

ax = axes[1]
colors = plt.cm.tab10(np.linspace(0, 1, len(pair_results)))
for (pair_name, data), color in zip(pair_results.items(), colors):
    t  = data["trades"]
    et = pd.to_datetime(t["exit_time"])
    ax.plot(et, t["net_pnl"].cumsum().values, label=pair_name, color=color, lw=1.5)
ax.axhline(0, color="black", lw=0.8)
ax.set_title("Per-pair Equity Curves")
ax.set_ylabel("Cumulative net P&L")
ax.legend(fontsize=8)

if spy_return is not None and n_rows == 3:
    ax = axes[2]
    ax2 = ax.twinx()

    # Strategy: normalise cumulative to % starting from 0
    first_trade_val = cumulative.values[0]
    strat_norm = (cumulative.values - first_trade_val) / max(abs(first_trade_val), 1) * 100

    ax.plot(exit_times, strat_norm, color="blue", lw=2, label="Strategy (normalised %)")
    ax2.plot(spy_cum.index, (spy_cum.values - 1) * 100, color="orange",
             lw=2, linestyle="--", label=f"SPY buy & hold")

    ax.axhline(0, color="black", lw=0.8)
    ax.set_ylabel("Strategy return (%)", color="blue")
    ax2.set_ylabel("SPY return (%)", color="orange")
    ax.set_title(f"Strategy vs SPY  |  Strategy Sharpe={sharpe:.2f}  "
                 f"SPY Sharpe={spy_sharpe:.2f}")

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "backtest_results.png", dpi=150)
plt.show()
print(f"Chart saved to {OUTPUT_DIR / 'backtest_results.png'}")

# ── Z-score debug plots per pair ──────────────────────────────────────────────
try:
    from step5f_debug_plot import plot_zscore_debug
    print("\nGenerating Z-score debug plots …")
    for pair_name, data in pair_results.items():
        t1, t2 = pair_name.split("-")
        ez, xz, sz = _opt_params.get(pair_name, (ENTRY_Z, EXIT_Z, STOP_Z))
        plot_zscore_debug(
            data["signals"], data["trades"],
            pair_name, entry_z=ez, exit_z=xz, stop_z=sz,
        )
except Exception as _e:
    print(f"Debug plots skipped: {_e}")


In [ ]:
# Quick summary of backtest results
if 'df_trades' in dir() and df_trades is not None and len(df_trades) > 0:
    print(f"Trades: {len(df_trades)}")
    display(df_trades[["pair","entry_time","exit_time","direction","exit_reason","net_pnl"]].head(10))
    print(f"\nNet P&L: {df_trades['net_pnl'].sum():+.4f}")
    print(f"Win rate: {(df_trades['net_pnl']>0).mean()*100:.1f}%")


## Step 5 — Analysis


### 5a — Exit Z Grid Search


In [ ]:
# Fallback: load df_trades from CSV if not in memory (e.g. running this cell standalone)
if 'df_trades' not in dir() or df_trades is None:
    import pandas as pd
    from config import DATA_DIR
    df_trades = pd.read_csv(DATA_DIR / "trades.csv")
    df_trades["exit_time"]  = pd.to_datetime(df_trades["exit_time"],  utc=True)
    df_trades["entry_time"] = pd.to_datetime(df_trades["entry_time"], utc=True)
    print(f"Loaded {len(df_trades)} trades from CSV")
else:
    print(f"Using in-memory df_trades: {len(df_trades)} trades")

# ── Step 5a: Exit Z Grid Search ─────────────────────────────────────────────
    CLOSES_FILE,
    ENTRY_Z, STOP_Z,
    COST_PER_SIDE, BORROW_RATE_ANNUAL,
    RTH_START, RTH_END, SIGNAL_START, RECENT_BARS,
    DATA_DIR, OUTPUT_DIR,
)

EXIT_Z_GRID = [0.5, 0.3, 0.1, 0.0, -0.1, -0.2, -0.3]
from config import BARS_PER_DAY
BARS_PER_TRADING_DAY = BARS_PER_DAY


def load_closes() -> pd.DataFrame:
    closes = pd.read_csv(DATA_DIR / CLOSES_FILE, index_col=0, parse_dates=True)
    if closes.index.tz is None:
        closes.index = closes.index.tz_localize("UTC").tz_convert("US/Eastern")
    else:
        closes.index = closes.index.tz_convert("US/Eastern")
    return closes.between_time(RTH_START, RTH_END).dropna().tail(RECENT_BARS)


def build_signals(closes, t1, t2, beta, half_life) -> pd.DataFrame:
    spread = closes[t1] - beta * closes[t2]
    window = max(20, min(int(half_life), 200))
    zscore = (spread - spread.rolling(window).mean()) / spread.rolling(window).std()
    return pd.DataFrame({
        f"{t1}_close": closes[t1],
        f"{t2}_close": closes[t2],
        "spread": spread,
        "zscore": zscore,
    }).dropna().between_time(SIGNAL_START, RTH_END)


def backtest_pair(df, t1, t2, beta, exit_z) -> pd.DataFrame:
    t1_col, t2_col = f"{t1}_close", f"{t2}_close"
    position = 0
    entry_spread = entry_t1 = entry_t2 = 0.0
    entry_bar = 0
    trades = []

    for i in range(len(df)):
        z          = df["zscore"].iloc[i]
        spread_now = df["spread"].iloc[i]
        t1_price   = df[t1_col].iloc[i]
        t2_price   = df[t2_col].iloc[i]

        if position != 0:
            exit_signal = (position == 1 and z > -exit_z) or (position == -1 and z < exit_z)
            stop_signal = (position == 1 and z < -STOP_Z) or (position == -1 and z > STOP_Z)

            if exit_signal or stop_signal:
                gross_pnl      = position * (spread_now - entry_spread)
                notional       = entry_t1 + beta * entry_t2
                tx_cost        = 2 * notional * COST_PER_SIDE
                holding_days   = (i - entry_bar) / BARS_PER_TRADING_DAY
                short_notional = (beta * entry_t2) if position == 1 else entry_t1
                borrow_cost    = short_notional * BORROW_RATE_ANNUAL * holding_days / 252
                trades.append(gross_pnl - tx_cost - borrow_cost)
                position = 0

        if position == 0:
            if z < -ENTRY_Z:
                position = 1
            elif z > ENTRY_Z:
                position = -1
            if position != 0:
                entry_spread = spread_now
                entry_t1     = t1_price
                entry_t2     = t2_price
                entry_bar    = i

    return trades


# ── Load ──────────────────────────────────────────────────────────────────────
closes = load_closes()
pairs  = pd.read_csv(DATA_DIR / "pairs_selected.csv")

if pairs.empty:
    print("pairs_selected.csv is empty — run step2_pairs.py first")  # non-fatal

print(f"Grid search EXIT_Z = {EXIT_Z_GRID}")
print(f"Pairs: {list(pairs['pair'])}\n")

# ── Grid search ───────────────────────────────────────────────────────────────
rows = []

for exit_z in EXIT_Z_GRID:
    all_pnl = []

    for _, row in pairs.iterrows():
        t1, t2    = row["pair"].split("-")
        beta      = row["beta"]
        half_life = row["half_life_bars"]

        if t1 not in closes.columns or t2 not in closes.columns:
            continue

        df_sig = build_signals(closes, t1, t2, beta, half_life)
        trade_pnl = backtest_pair(df_sig, t1, t2, beta, exit_z)
        all_pnl.extend(trade_pnl)

    if not all_pnl:
        continue

    pnl   = pd.Series(all_pnl)
    n     = len(pnl)
    curve = pnl.cumsum()
    wr    = (pnl > 0).mean() * 100
    tpy   = n / (closes.index[-1] - closes.index[0]).days * 365.25
    sh    = pnl.mean() / pnl.std() * np.sqrt(tpy) if pnl.std() > 0 else 0.0
    dd    = (curve - curve.cummax()).min()

    rows.append({
        "exit_z":    exit_z,
        "trades":    n,
        "win_rate":  wr,
        "sharpe":    sh,
        "total_pnl": pnl.sum(),
        "avg_pnl":   pnl.mean(),
        "max_dd":    dd,
        "curve":     curve.values,
    })

# ── Table ─────────────────────────────────────────────────────────────────────
print("=" * 70)
print(f"{'exit_z':>8} {'trades':>7} {'win_rate':>9} {'sharpe':>7} "
      f"{'total_pnl':>11} {'avg_pnl':>9} {'max_dd':>10}")
print("=" * 70)
for r in rows:
    print(f"{r['exit_z']:>+8.1f} {r['trades']:>7} {r['win_rate']:>8.1f}% "
          f"{r['sharpe']:>7.2f} {r['total_pnl']:>+11.4f} "
          f"{r['avg_pnl']:>+9.4f} {r['max_dd']:>10.4f}")

best = max(rows, key=lambda r: r["sharpe"])
print(f"\nBest EXIT_Z by Sharpe: {best['exit_z']:+.1f}  →  "
      f"Sharpe={best['sharpe']:.2f}, WR={best['win_rate']:.1f}%, "
      f"Trades={best['trades']}, P&L={best['total_pnl']:+.4f}")

# ── Charts ────────────────────────────────────────────────────────────────────
OUTPUT_DIR.mkdir(exist_ok=True)
colors = plt.cm.RdYlGn(np.linspace(0.1, 0.9, len(rows)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

for r, color in zip(rows, colors):
    lw    = 2.5 if r["exit_z"] == best["exit_z"] else 1.2
    alpha = 1.0 if r["exit_z"] == best["exit_z"] else 0.6
    label = f"exit_z={r['exit_z']:+.1f}  Sh={r['sharpe']:.2f}  WR={r['win_rate']:.0f}%"
    ax1.plot(r["curve"], label=label, color=color, lw=lw, alpha=alpha)
ax1.axhline(0, color="black", lw=0.8)
ax1.set_title("Portfolio Equity Curves by EXIT_Z")
ax1.set_xlabel("Trade #")
ax1.set_ylabel("Cumulative net P&L")
ax1.legend(fontsize=7)

sharpes   = [r["sharpe"]  for r in rows]
exit_zs   = [f"{r['exit_z']:+.1f}" for r in rows]
bars = ax2.bar(exit_zs, sharpes, color=colors, edgecolor="white")
ax2.axhline(0, color="black", lw=0.8)
ax2.set_title("Sharpe by EXIT_Z (all pairs combined)")
ax2.set_xlabel("EXIT_Z")
ax2.set_ylabel("Sharpe")
for bar, val in zip(bars, sharpes):
    ax2.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 0.02 * max(sharpes),
             f"{val:.2f}", ha="center", va="bottom", fontsize=9)

plt.suptitle("EXIT_Z Grid Search — All Pairs Combined", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "exit_z_grid.png", dpi=150)
plt.show()
print(f"\nChart saved to {OUTPUT_DIR / 'exit_z_grid.png'}")
# plt.show()


### 5b — Sniper Grid Search


In [ ]:
# Fallback: load df_trades from CSV if not in memory (e.g. running this cell standalone)
if 'df_trades' not in dir() or df_trades is None:
    import pandas as pd
    from config import DATA_DIR
    df_trades = pd.read_csv(DATA_DIR / "trades.csv")
    df_trades["exit_time"]  = pd.to_datetime(df_trades["exit_time"],  utc=True)
    df_trades["entry_time"] = pd.to_datetime(df_trades["entry_time"], utc=True)
    print(f"Loaded {len(df_trades)} trades from CSV")
else:
    print(f"Using in-memory df_trades: {len(df_trades)} trades")

# ── Step 5b: Sniper Grid Search ─────────────────────────────────────────────
    CLOSES_FILE,
    STOP_Z as DEFAULT_STOP_Z,
    COST_PER_SIDE, BORROW_RATE_ANNUAL,
    RTH_START, RTH_END, SIGNAL_START, RECENT_BARS,
    DATA_DIR, OUTPUT_DIR,
)

ENTRY_Z_GRID = [3.5, 3.3, 3.2, 3.0]
EXIT_Z_GRID  = [0.3, 0.2, 0.1, 0.0, -0.1, -0.2, -0.3]
STOP_Z_GRID  = [3.8, 4.2, 4.4]

BARS_PER_DAY = 26
COMBOS       = list(itertools.product(ENTRY_Z_GRID, EXIT_Z_GRID, STOP_Z_GRID))
print(f"Sniper grid: {len(ENTRY_Z_GRID)} entry × {len(EXIT_Z_GRID)} exit × "
      f"{len(STOP_Z_GRID)} stop = {len(COMBOS)} combinations\n")


def load_closes() -> pd.DataFrame:
    closes = pd.read_csv(DATA_DIR / CLOSES_FILE, index_col=0, parse_dates=True)
    if closes.index.tz is None:
        closes.index = closes.index.tz_localize("UTC").tz_convert("US/Eastern")
    else:
        closes.index = closes.index.tz_convert("US/Eastern")
    return closes.between_time(RTH_START, RTH_END).dropna().tail(RECENT_BARS)


def build_signals(closes, t1, t2, beta, half_life) -> pd.DataFrame:
    spread = closes[t1] - beta * closes[t2]
    window = max(20, min(int(half_life), 200))
    zscore = (spread - spread.rolling(window).mean()) / spread.rolling(window).std()
    return pd.DataFrame({
        f"{t1}_close": closes[t1],
        f"{t2}_close": closes[t2],
        "spread": spread,
        "zscore": zscore,
    }).dropna().between_time(SIGNAL_START, RTH_END)


def backtest(df, t1, t2, beta, entry_z, exit_z, stop_z) -> list:
    t1_col, t2_col = f"{t1}_close", f"{t2}_close"
    position = 0
    entry_spread = entry_t1 = entry_t2 = 0.0
    entry_bar = 0
    pnls = []

    for i in range(len(df)):
        z          = df["zscore"].iloc[i]
        spread_now = df["spread"].iloc[i]
        t1_price   = df[t1_col].iloc[i]
        t2_price   = df[t2_col].iloc[i]

        if position != 0:
            exit_sig = (position == 1 and z > -exit_z) or (position == -1 and z < exit_z)
            stop_sig = (position == 1 and z < -stop_z) or (position == -1 and z > stop_z)

            if exit_sig or stop_sig:
                gross          = position * (spread_now - entry_spread)
                notional       = entry_t1 + beta * entry_t2
                tx_cost        = 2 * notional * COST_PER_SIDE
                holding_days   = (i - entry_bar) / BARS_PER_DAY
                short_notional = (beta * entry_t2) if position == 1 else entry_t1
                borrow_cost    = short_notional * BORROW_RATE_ANNUAL * holding_days / 252
                pnls.append(gross - tx_cost - borrow_cost)
                position = 0

        if position == 0:
            if z < -entry_z:
                position = 1
            elif z > entry_z:
                position = -1
            if position != 0:
                entry_spread = spread_now
                entry_t1     = t1_price
                entry_t2     = t2_price
                entry_bar    = i

    return pnls


# ── Load ──────────────────────────────────────────────────────────────────────
closes = load_closes()
pairs  = pd.read_csv(DATA_DIR / "pairs_selected.csv")

if pairs.empty:
    print("pairs_selected.csv is empty — run step2_pairs.py first")  # non-fatal

# Pre-build signals for each pair
signals = {}
for _, row in pairs.iterrows():
    t1, t2 = row["pair"].split("-")
    if t1 not in closes.columns or t2 not in closes.columns:
        continue
    signals[row["pair"]] = (
        build_signals(closes, t1, t2, row["beta"], row["half_life_bars"]),
        t1, t2, row["beta"]
    )

print(f"Pairs: {list(signals.keys())}")
print(f"Running {len(COMBOS)} combinations...\n")

days_total = (closes.index[-1] - closes.index[0]).days

# ── Grid search ───────────────────────────────────────────────────────────────
results = []

for entry_z, exit_z, stop_z in COMBOS:
    all_pnl = []

    for pair_name, (df_sig, t1, t2, beta) in signals.items():
        all_pnl.extend(backtest(df_sig, t1, t2, beta, entry_z, exit_z, stop_z))

    if not all_pnl:
        continue

    pnl   = np.array(all_pnl)
    n     = len(pnl)
    curve = np.cumsum(pnl)
    wr    = (pnl > 0).mean() * 100
    tpy   = n / (days_total / 365.25)
    sh    = pnl.mean() / pnl.std() * np.sqrt(tpy) if pnl.std() > 0 else 0.0
    dd    = (curve - np.maximum.accumulate(curve)).min()
    pf_w  = pnl[pnl > 0].sum()
    pf_l  = abs(pnl[pnl <= 0].sum())
    pf    = pf_w / pf_l if pf_l > 0 else float("inf")

    results.append({
        "entry_z":   entry_z,
        "exit_z":    exit_z,
        "stop_z":    stop_z,
        "trades":    n,
        "win_rate":  wr,
        "sharpe":    sh,
        "total_pnl": pnl.sum(),
        "avg_pnl":   pnl.mean(),
        "max_dd":    dd,
        "pf":        pf,
        "curve":     curve,
    })

df_res = pd.DataFrame([{k: v for k, v in r.items() if k != "curve"}
                        for r in results])
df_res = df_res.sort_values("sharpe", ascending=False).reset_index(drop=True)

# ── Print full table ──────────────────────────────────────────────────────────
print("=" * 85)
print(f"{'#':>3} {'entry':>6} {'exit':>6} {'stop':>6} {'trades':>7} "
      f"{'WR':>6} {'sharpe':>7} {'total_pnl':>11} {'max_dd':>9} {'PF':>6}")
print("=" * 85)

for i, row in df_res.iterrows():
    marker = " ◄ BEST" if i == 0 else (" ◄ top3" if i < 3 else "")
    print(f"{i+1:>3} {row['entry_z']:>6.1f} {row['exit_z']:>+6.1f} {row['stop_z']:>6.1f} "
          f"{row['trades']:>7.0f} {row['win_rate']:>5.1f}% {row['sharpe']:>7.2f} "
          f"{row['total_pnl']:>+11.4f} {row['max_dd']:>9.4f} {row['pf']:>6.2f}{marker}")

best = df_res.iloc[0]
print(f"\n{'='*85}")
print(f"BEST: entry={best['entry_z']}  exit={best['exit_z']:+.1f}  stop={best['stop_z']}  "
      f"→  Sharpe={best['sharpe']:.2f}  WR={best['win_rate']:.1f}%  "
      f"Trades={best['trades']:.0f}  P&L={best['total_pnl']:+.4f}")

# ── Heatmaps: Sharpe by entry_z × exit_z for each stop_z ─────────────────────
OUTPUT_DIR.mkdir(exist_ok=True)

fig, axes = plt.subplots(1, len(STOP_Z_GRID), figsize=(6 * len(STOP_Z_GRID), 5))
if len(STOP_Z_GRID) == 1:
    axes = [axes]

for ax, stop_z in zip(axes, STOP_Z_GRID):
    sub = df_res[df_res["stop_z"] == stop_z]
    pivot = sub.pivot(index="entry_z", columns="exit_z", values="sharpe")
    pivot = pivot.reindex(index=sorted(pivot.index, reverse=True))

    im = ax.imshow(pivot.values, cmap="RdYlGn", aspect="auto",
                   vmin=df_res["sharpe"].min(), vmax=df_res["sharpe"].max())
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f"{v:+.1f}" for v in pivot.columns], fontsize=9)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f"{v:.1f}" for v in pivot.index], fontsize=9)

    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                        fontsize=8, fontweight="bold",
                        color="white" if abs(val) > df_res["sharpe"].std() else "black")

    ax.set_title(f"Sharpe  |  stop_z = {stop_z}", fontsize=11, fontweight="bold")
    ax.set_xlabel("EXIT_Z")
    ax.set_ylabel("ENTRY_Z")
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle("Sniper Grid — Sharpe Heatmap (entry × exit × stop)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "sniper_grid.png", dpi=150)
plt.show()
print(f"\nHeatmap saved to {OUTPUT_DIR / 'sniper_grid.png'}")

# ── Top-10 equity curves ──────────────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(14, 6))
top10     = df_res.head(10)
colors    = plt.cm.RdYlGn(np.linspace(0.2, 0.9, 10))

result_map = {(r["entry_z"], r["exit_z"], r["stop_z"]): r for r in results}

for rank, (_, row) in enumerate(zip(range(10), top10.itertuples())):
    r = result_map[(row.entry_z, row.exit_z, row.stop_z)]
    color = colors[rank]
    lbl = (f"#{rank+1} entry={r['entry_z']} exit={r['exit_z']:+.1f} "
           f"stop={r['stop_z']}  Sh={r['sharpe']:.2f}")
    lw  = 2.5 if rank == 0 else 1.2
    ax2.plot(r["curve"], label=lbl, color=color, lw=lw, alpha=0.85)

ax2.axhline(0, color="black", lw=0.8)
ax2.set_title("Top-10 Sniper Combinations — Equity Curves", fontsize=12, fontweight="bold")
ax2.set_xlabel("Trade #")
ax2.set_ylabel("Cumulative net P&L")
ax2.legend(fontsize=7, loc="upper left")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "sniper_top10.png", dpi=150)
plt.show()
print(f"Top-10 curves saved to {OUTPUT_DIR / 'sniper_top10.png'}")
# plt.show()


### 5c — Bootstrap Monte Carlo


In [ ]:
# Fallback: load df_trades from CSV if not in memory (e.g. running this cell standalone)
if 'df_trades' not in dir() or df_trades is None:
    import pandas as pd
    from config import DATA_DIR
    df_trades = pd.read_csv(DATA_DIR / "trades.csv")
    df_trades["exit_time"]  = pd.to_datetime(df_trades["exit_time"],  utc=True)
    df_trades["entry_time"] = pd.to_datetime(df_trades["entry_time"], utc=True)
    print(f"Loaded {len(df_trades)} trades from CSV")
else:
    print(f"Using in-memory df_trades: {len(df_trades)} trades")

# ── Step 5c: Bootstrap Monte Carlo ──────────────────────────────────────────
N_SIMS = 10_000


pnl = df_trades["net_pnl"].values
n = len(pnl)

real_curve = np.cumsum(pnl)
real_total = real_curve[-1]
real_sharpe = pnl.mean() / pnl.std() * np.sqrt(n) if pnl.std() > 0 else 0.0
real_dd = (real_curve - np.maximum.accumulate(real_curve)).min()

print(f"Loaded {n} trades. Running {N_SIMS:,} simulations...\n")

# ── Bootstrap resampling ──────────────────────────────────────────────────────
# Resample trades with replacement → N possible equity curves
idx = np.random.randint(0, n, size=(N_SIMS, n))
boot_samples = pnl[idx]                                   # (N_SIMS, n)
boot_curves  = np.cumsum(boot_samples, axis=1)            # (N_SIMS, n)
boot_totals  = boot_curves[:, -1]
boot_sharpes = (boot_samples.mean(axis=1)
                / boot_samples.std(axis=1)
                * np.sqrt(n))
running_max  = np.maximum.accumulate(boot_curves, axis=1)
boot_drawdowns = (boot_curves - running_max).min(axis=1)

# ── Random sign test ──────────────────────────────────────────────────────────
# Keep trade magnitudes, randomly flip signs → baseline of pure chance
signs        = np.random.choice([-1, 1], size=(N_SIMS, n))
sign_samples = np.abs(pnl) * signs
sign_totals  = sign_samples.sum(axis=1)
sign_sharpes = (sign_samples.mean(axis=1)
                / sign_samples.std(axis=1)
                * np.sqrt(n))

p_value_total  = np.mean(sign_totals  >= real_total)
p_value_sharpe = np.mean(sign_sharpes >= real_sharpe)

# ── Print results ─────────────────────────────────────────────────────────────
def ci(arr, lo=5, hi=95):
    return np.percentile(arr, lo), np.percentile(arr, hi)

print("=" * 60)
print("MONTE CARLO RESULTS")
print("=" * 60)

print(f"\nReal strategy:")
print(f"  Total P&L : {real_total:+.4f}")
print(f"  Sharpe    : {real_sharpe:.2f}")
print(f"  Max DD    : {real_dd:.4f}")

lo, hi = ci(boot_totals)
print(f"\nBootstrap 90% CI  ({N_SIMS:,} samples with replacement):")
print(f"  Total P&L : [{lo:+.4f}, {hi:+.4f}]")
lo, hi = ci(boot_sharpes)
print(f"  Sharpe    : [{lo:.2f}, {hi:.2f}]")
lo, hi = ci(boot_drawdowns)
print(f"  Max DD    : [{lo:.4f}, {hi:.4f}]")

print(f"\nRandom sign test  (is the result non-random?):")
sig_total  = "SIGNIFICANT" if p_value_total  < 0.05 else "not significant"
sig_sharpe = "SIGNIFICANT" if p_value_sharpe < 0.05 else "not significant"
print(f"  p (P&L)   : {p_value_total:.4f}  → {sig_total}")
print(f"  p (Sharpe): {p_value_sharpe:.4f}  → {sig_sharpe}")

# ── Visualisation ─────────────────────────────────────────────────────────────
OUTPUT_DIR.mkdir(exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
x = np.arange(1, n + 1)

# ── 1. Equity curve fan (the "branches") ─────────────────────────────────────
ax = axes[0, 0]

# Draw 300 individual paths as light branches
sample_idx = np.random.choice(N_SIMS, size=300, replace=False)
for i in sample_idx:
    ax.plot(x, boot_curves[i], color="steelblue", alpha=0.04, lw=0.8)

# Percentile bands
p5, p25, p50, p75, p95 = np.percentile(boot_curves, [5, 25, 50, 75, 95], axis=0)
ax.fill_between(x, p5,  p95, alpha=0.15, color="steelblue", label="5–95%")
ax.fill_between(x, p25, p75, alpha=0.35, color="steelblue", label="25–75%")
ax.plot(x, p50, color="steelblue", lw=1.5, linestyle="--", label="Median path")
ax.plot(x, real_curve, color="red", lw=2.5, label="Real strategy", zorder=5)
ax.axhline(0, color="black", lw=0.8)

ax.set_title("Equity Curve — Bootstrap Fan (branches)")
ax.set_xlabel("Trade #")
ax.set_ylabel("Cumulative P&L")
ax.legend(fontsize=8)

# ── 2. Distribution of final P&L ─────────────────────────────────────────────
ax = axes[0, 1]
ax.hist(boot_totals, bins=80, color="steelblue", alpha=0.7, edgecolor="none")
ax.axvline(real_total, color="red",    lw=2,   label=f"Real: {real_total:+.3f}")
ax.axvline(np.percentile(boot_totals, 5),  color="orange", lw=1.5,
           linestyle="--", label="5th pct")
ax.axvline(np.percentile(boot_totals, 95), color="green",  lw=1.5,
           linestyle="--", label="95th pct")
ax.axvline(0, color="black", lw=0.8)
ax.set_title("Distribution of Final P&L (bootstrap)")
ax.set_xlabel("Total P&L")
ax.legend(fontsize=8)

# ── 3. Distribution of Sharpe ─────────────────────────────────────────────────
ax = axes[1, 0]
ax.hist(boot_sharpes, bins=80, color="mediumseagreen", alpha=0.7, edgecolor="none")
ax.axvline(real_sharpe, color="red", lw=2, label=f"Real: {real_sharpe:.2f}")
ax.axvline(0, color="black", lw=0.8)
ax.set_title("Distribution of Sharpe Ratio (bootstrap)")
ax.set_xlabel("Sharpe")
ax.legend(fontsize=8)

# ── 4. Random sign test ───────────────────────────────────────────────────────
ax = axes[1, 1]
ax.hist(sign_totals, bins=80, color="gray", alpha=0.6, edgecolor="none",
        label="Random (sign-flipped)")
ax.axvline(real_total, color="red", lw=2,
           label=f"Real: {real_total:+.3f}  (p={p_value_total:.3f})")
ax.axvline(0, color="black", lw=0.8)
ax.set_title("Random Sign Test — is the result luck?")
ax.set_xlabel("Total P&L")
ax.legend(fontsize=8)

plt.suptitle(
    f"Monte Carlo Analysis  |  {n} trades  |  {N_SIMS:,} simulations",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "montecarlo_results.png", dpi=150)
plt.show()
print(f"\nChart saved to {OUTPUT_DIR / 'montecarlo_results.png'}")
# plt.show()


### 5d — Dashboard


In [ ]:
# Fallback: load df_trades from CSV if not in memory (e.g. running this cell standalone)
if 'df_trades' not in dir() or df_trades is None:
    import pandas as pd
    from config import DATA_DIR
    df_trades = pd.read_csv(DATA_DIR / "trades.csv")
    df_trades["exit_time"]  = pd.to_datetime(df_trades["exit_time"],  utc=True)
    df_trades["entry_time"] = pd.to_datetime(df_trades["entry_time"], utc=True)
    print(f"Loaded {len(df_trades)} trades from CSV")
else:
    print(f"Using in-memory df_trades: {len(df_trades)} trades")

# ── Step 5d: Dashboard ───────────────────────────────────────────────────────
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from config import DATA_DIR, OUTPUT_DIR
ROLLING_WINDOW = 20

df = df_trades.sort_values("exit_time").reset_index(drop=True).copy()
df["exit_time"]  = pd.to_datetime(df["exit_time"],  utc=True)
df["entry_time"] = pd.to_datetime(df["entry_time"], utc=True)
ROLLING_WINDOW = 20   # trades for rolling Sharpe

# ── Load ──────────────────────────────────────────────────────────────────────


df = df.sort_values("exit_time").reset_index(drop=True)
pnl        = df["net_pnl"]
cumulative = pnl.cumsum()
drawdown   = cumulative - cumulative.cummax()
pairs      = df["pair"].unique()
exit_times = df["exit_time"]

days_total      = (df["exit_time"].iloc[-1] - df["exit_time"].iloc[0]).days
trades_per_year = len(df) / (days_total / 365.25)
sharpe          = pnl.mean() / pnl.std() * np.sqrt(trades_per_year) if pnl.std() > 0 else 0
win_rate        = (pnl > 0).mean() * 100
profit_factor   = (pnl[pnl > 0].sum() / abs(pnl[pnl <= 0].sum())
                   if pnl[pnl <= 0].sum() != 0 else float("inf"))
max_dd          = drawdown.min()
total_costs     = (df["tx_cost"] + df["borrow_cost"]).sum()

print(f"Dashboard: {len(df)} trades across {len(pairs)} pairs")

# ── Layout ────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(20, 26))
fig.patch.set_facecolor("#f8f9fa")

gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.45, wspace=0.35)

TITLE_SIZE = 10
colors_pairs = plt.cm.tab10(np.linspace(0, 1, len(pairs)))
pair_color   = {p: c for p, c in zip(pairs, colors_pairs)}

# ═════════════════════════════════════════════════════════════════════════════
# ROW 1
# ═════════════════════════════════════════════════════════════════════════════

# 1. Portfolio equity curve
ax = fig.add_subplot(gs[0, :2])
ax.plot(exit_times, cumulative.values, color="#2196F3", lw=2, label="Net P&L")
ax.plot(exit_times, df["gross_pnl"].cumsum().values,
        color="#2196F3", lw=1, linestyle="--", alpha=0.4, label="Gross P&L")
ax.fill_between(exit_times, cumulative.values, 0,
                where=(cumulative.values >= 0), color="#4CAF50", alpha=0.1)
ax.fill_between(exit_times, cumulative.values, 0,
                where=(cumulative.values < 0),  color="#F44336", alpha=0.1)
ax.axhline(0, color="black", lw=0.8)
ax.set_title("Portfolio Equity Curve (net vs gross)", fontsize=TITLE_SIZE, fontweight="bold")
ax.set_ylabel("Cumulative P&L")
ax.legend(fontsize=8)
ax.set_facecolor("#ffffff")

# 2. Per-pair equity
ax = fig.add_subplot(gs[0, 2])
for pair in pairs:
    t = df[df["pair"] == pair]
    et = pd.to_datetime(t["exit_time"])
    ax.plot(et, t["net_pnl"].cumsum().values,
            label=pair, color=pair_color[pair], lw=1.5)
ax.axhline(0, color="black", lw=0.8)
ax.set_title("Per-pair Equity", fontsize=TITLE_SIZE, fontweight="bold")
ax.legend(fontsize=7)
ax.set_facecolor("#ffffff")

# ═════════════════════════════════════════════════════════════════════════════
# ROW 2
# ═════════════════════════════════════════════════════════════════════════════

# 3. P&L distribution
ax = fig.add_subplot(gs[1, 0])
wins   = pnl[pnl > 0]
losses = pnl[pnl <= 0]
ax.hist(losses, bins=25, color="#F44336", alpha=0.7, label=f"Losses ({len(losses)})")
ax.hist(wins,   bins=25, color="#4CAF50", alpha=0.7, label=f"Wins ({len(wins)})")
ax.axvline(pnl.mean(), color="black", lw=1.5, linestyle="--",
           label=f"Mean {pnl.mean():+.3f}")
ax.axvline(0, color="black", lw=0.8)
ax.set_title("Trade P&L Distribution", fontsize=TITLE_SIZE, fontweight="bold")
ax.set_xlabel("Net P&L per trade")
ax.legend(fontsize=7)
ax.set_facecolor("#ffffff")

# 4. Drawdown
ax = fig.add_subplot(gs[1, 1])
ax.fill_between(exit_times, drawdown.values, 0, color="#F44336", alpha=0.6)
ax.plot(exit_times, drawdown.values, color="#B71C1C", lw=1)
ax.axhline(max_dd, color="darkred", lw=1, linestyle="--",
           label=f"Max DD {max_dd:.3f}")
ax.set_title("Drawdown Curve", fontsize=TITLE_SIZE, fontweight="bold")
ax.set_ylabel("Drawdown")
ax.legend(fontsize=8)
ax.set_facecolor("#ffffff")

# 5. Rolling Sharpe
ax = fig.add_subplot(gs[1, 2])
rolling_sh = [
    (pnl.iloc[max(0, i - ROLLING_WINDOW):i].mean()
     / pnl.iloc[max(0, i - ROLLING_WINDOW):i].std()
     * np.sqrt(ROLLING_WINDOW)
     if pnl.iloc[max(0, i - ROLLING_WINDOW):i].std() > 0 else 0)
    for i in range(1, len(pnl) + 1)
]
ax.plot(exit_times, rolling_sh, color="#9C27B0", lw=1.5)
ax.axhline(0, color="black", lw=0.8)
ax.axhline(1, color="green", lw=0.8, linestyle="--", alpha=0.5)
ax.fill_between(exit_times, rolling_sh, 0,
                where=np.array(rolling_sh) >= 0, color="#4CAF50", alpha=0.15)
ax.fill_between(exit_times, rolling_sh, 0,
                where=np.array(rolling_sh) < 0,  color="#F44336", alpha=0.15)
ax.set_title(f"Rolling Sharpe ({ROLLING_WINDOW} trades)", fontsize=TITLE_SIZE, fontweight="bold")
ax.set_facecolor("#ffffff")

# ═════════════════════════════════════════════════════════════════════════════
# ROW 3
# ═════════════════════════════════════════════════════════════════════════════

# 6. Win rate per pair
ax = fig.add_subplot(gs[2, 0])
pair_wr = {p: (df[df["pair"] == p]["net_pnl"] > 0).mean() * 100 for p in pairs}
bar_colors = ["#4CAF50" if v >= 50 else "#F44336" for v in pair_wr.values()]
bars = ax.bar(pair_wr.keys(), pair_wr.values(), color=bar_colors, edgecolor="white")
ax.axhline(50, color="black", lw=1, linestyle="--")
ax.set_title("Win Rate per Pair", fontsize=TITLE_SIZE, fontweight="bold")
ax.set_ylabel("Win rate (%)")
ax.set_ylim(0, 100)
for bar, val in zip(bars, pair_wr.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f"{val:.0f}%", ha="center", va="bottom", fontsize=8)
ax.set_facecolor("#ffffff")

# 7. Hold time distribution
ax = fig.add_subplot(gs[2, 1])
bars_per_day = 26
hold_days = df["holding_bars"] / bars_per_day
ax.hist(hold_days, bins=30, color="#FF9800", alpha=0.8, edgecolor="white")
ax.axvline(hold_days.mean(), color="black", lw=1.5, linestyle="--",
           label=f"Mean {hold_days.mean():.1f}d")
ax.set_title("Holding Time Distribution", fontsize=TITLE_SIZE, fontweight="bold")
ax.set_xlabel("Days in trade")
ax.legend(fontsize=8)
ax.set_facecolor("#ffffff")

# 8. Long vs Short P&L
ax = fig.add_subplot(gs[2, 2])
directions = df.groupby("direction")["net_pnl"].agg(["sum", "count", "mean"])
dir_colors = {"LONG": "#4CAF50", "SHORT": "#2196F3"}
for i, (direction, row_d) in enumerate(directions.iterrows()):
    color = dir_colors.get(direction, "gray")
    ax.bar(direction, row_d["sum"], color=color, alpha=0.8, label=f"{direction} ({int(row_d['count'])})")
    ax.text(i, row_d["sum"] + (1 if row_d["sum"] >= 0 else -3),
            f"{row_d['sum']:+.1f}", ha="center", fontsize=8)
ax.axhline(0, color="black", lw=0.8)
ax.set_title("P&L by Direction", fontsize=TITLE_SIZE, fontweight="bold")
ax.set_ylabel("Total net P&L")
ax.legend(fontsize=8)
ax.set_facecolor("#ffffff")

# ═════════════════════════════════════════════════════════════════════════════
# ROW 4
# ═════════════════════════════════════════════════════════════════════════════

# 9. Monthly P&L heatmap
ax = fig.add_subplot(gs[3, :2])
df["month"]    = df["exit_time"].dt.to_period("M")
monthly_pnl    = df.groupby(["pair", "month"])["net_pnl"].sum().unstack(fill_value=0)
monthly_pnl.columns = monthly_pnl.columns.astype(str)

cmap = LinearSegmentedColormap.from_list("rg", ["#F44336", "white", "#4CAF50"])
im = ax.imshow(monthly_pnl.values, aspect="auto", cmap=cmap,
               vmin=-abs(monthly_pnl.values).max(),
               vmax=abs(monthly_pnl.values).max())
ax.set_xticks(range(len(monthly_pnl.columns)))
ax.set_xticklabels(monthly_pnl.columns, rotation=45, ha="right", fontsize=7)
ax.set_yticks(range(len(monthly_pnl.index)))
ax.set_yticklabels(monthly_pnl.index, fontsize=8)
for i in range(monthly_pnl.shape[0]):
    for j in range(monthly_pnl.shape[1]):
        val = monthly_pnl.values[i, j]
        if val != 0:
            ax.text(j, i, f"{val:.1f}", ha="center", va="center",
                    fontsize=7, color="black")
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Monthly P&L Heatmap (per pair)", fontsize=TITLE_SIZE, fontweight="bold")
ax.set_facecolor("#ffffff")

# 10. Costs breakdown + summary metrics
ax = fig.add_subplot(gs[3, 2])
ax.axis("off")

gross_total  = df["gross_pnl"].sum()
tx_total     = df["tx_cost"].sum()
borrow_total = df["borrow_cost"].sum()
net_total    = pnl.sum()
stops        = (df["exit_reason"] == "STOP").sum()

metrics = [
    ("━━━ PORTFOLIO SUMMARY ━━━", "", "bold"),
    ("", "", "normal"),
    ("Trades",        f"{len(df)}  ({trades_per_year:.0f}/yr)", "normal"),
    ("Win rate",      f"{win_rate:.1f}%", "normal"),
    ("Stops",         f"{stops}", "normal"),
    ("Avg hold",      f"{(df['holding_bars']/26).mean():.1f} days", "normal"),
    ("", "", "normal"),
    ("━━━ P&L ━━━", "", "bold"),
    ("", "", "normal"),
    ("Gross P&L",     f"{gross_total:+.2f}", "normal"),
    ("  tx costs",    f"−{tx_total:.2f}", "normal"),
    ("  borrow",      f"−{borrow_total:.2f}", "normal"),
    ("Net P&L",       f"{net_total:+.2f}", "bold"),
    ("", "", "normal"),
    ("━━━ RISK ━━━", "", "bold"),
    ("", "", "normal"),
    ("Sharpe",        f"{sharpe:.2f}", "normal"),
    ("Profit factor", f"{profit_factor:.2f}", "normal"),
    ("Max drawdown",  f"{max_dd:.3f}", "normal"),
    ("Cost drag",     f"{total_costs/gross_total*100:.0f}% of gross", "normal"),
]

y = 0.98
for label, value, weight in metrics:
    if label.startswith("━"):
        ax.text(0.05, y, label, transform=ax.transAxes, fontsize=8,
                fontweight="bold", color="#333333", va="top")
    else:
        ax.text(0.05, y, label,  transform=ax.transAxes, fontsize=8.5,
                fontweight=weight, color="#555555", va="top")
        ax.text(0.70, y, value,  transform=ax.transAxes, fontsize=8.5,
                fontweight=weight, color="#111111", va="top", ha="right")
    y -= 0.05

ax.set_facecolor("#f0f4f8")
ax.set_title("Key Metrics", fontsize=TITLE_SIZE, fontweight="bold")

# ── Save ──────────────────────────────────────────────────────────────────────
OUTPUT_DIR.mkdir(exist_ok=True)
plt.suptitle("AFES — Statistical Arbitrage Dashboard",
             fontsize=16, fontweight="bold", y=0.995, color="#1a237e")

out_path = OUTPUT_DIR / "dashboard.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print(f"Dashboard saved to {out_path}")
# plt.show()


### 5e — Stress Test


In [ ]:
# ── Step 5e: Stress Test ────────────────────────────────────────────────────
# Note: stress test runs its own intraday backtest per window — no df_trades dependency.
    COST_MAKER, COST_TAKER, CIRCUIT_BREAKER_Z,
    CLOSES_FILE, ENTRY_Z, EXIT_Z, STOP_Z, ENTRY_Z_VOLATILE,
    BORROW_RATE_ANNUAL,
    RTH_START, RTH_END, SIGNAL_START,
    BARS_PER_DAY, DATA_DIR, OUTPUT_DIR,
    HURST_ENTRY_WINDOW,
)
from filters import HurstFilter
from step3j_wfo import check_coint_johansen, compute_half_life

WINDOW_MONTHS  = 6    # walk-forward window size
STEP_MONTHS    = 2    # how far each window advances

VIX_LOW    = 15.0
VIX_HIGH   = 25.0

OUTPUT_DIR.mkdir(exist_ok=True)


# ── Data loading ──────────────────────────────────────────────────────────────

def load_daily() -> pd.DataFrame:
    path = DATA_DIR / "closes_daily.csv"
    if not path.exists(): return None
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index, utc=True)
    if df.index.tz is None:
        df.index = df.index.tz_localize("UTC").tz_convert("US/Eastern")
    else:
        df.index = df.index.tz_convert("US/Eastern")
    return df

def load_closes() -> pd.DataFrame:
    path = DATA_DIR / CLOSES_FILE
    if not path.exists():
        path = DATA_DIR / "closes_15min.csv"
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index, utc=True)
    
    if df.index.tz is None:
        df.index = df.index.tz_localize("UTC").tz_convert("US/Eastern")
    else:
        df.index = df.index.tz_convert("US/Eastern")
    return df.between_time(RTH_START, RTH_END)


def load_vix_daily() -> pd.Series | None:
    iv_path = DATA_DIR / "iv_filter.csv"
    if not iv_path.exists():
        return None
    df = pd.read_csv(iv_path, index_col=0, parse_dates=True)
    if "vix" in df.columns:
        return df["vix"].dropna()
    return None


# ── Backtest engine (lean, no sizing) ────────────────────────────────────────

def run_backtest(df: pd.DataFrame, t1: str, t2: str, beta: float,
                 entry_z: float, exit_z: float, stop_z: float,
                 hurst_filter: HurstFilter = None,
                 spread_daily: pd.Series = None) -> list[dict]:
    t1c, t2c = f"{t1}_close", f"{t2}_close"
    pos = entry_spread = entry_t1 = entry_t2 = entry_sma = entry_std = 0.0
    entry_bar = 0
    trades = []
    pair_blocked = False

    for i in range(len(df)):
        if pair_blocked: continue
        z  = df["zscore"].iloc[i]
        s  = df["spread"].iloc[i]
        p1 = df[t1c].iloc[i]
        p2 = df[t2c].iloc[i]
        
        z_active = z
        if pos != 0 and entry_std > 0:
            z_active = (s - entry_sma) / entry_std

        if pos != 0:
            ex = (pos == 1 and z_active >= exit_z)  or (pos == -1 and z_active <= -exit_z)
            st = (pos == 1 and z_active <= -stop_z) or (pos == -1 and z_active >= stop_z)
            cb = abs(z_active) >= CIRCUIT_BREAKER_Z
            if ex or st or cb:
                if cb:
                    pair_blocked = True
                    st = True
                gross      = pos * (s - entry_spread)
                notional   = entry_t1 + beta * entry_t2
                tx         = notional * COST_MAKER + notional * (COST_MAKER if ex and not cb else COST_TAKER)
                hold_days  = (i - entry_bar) / BARS_PER_DAY
                borrow     = (beta * entry_t2 if pos == 1 else entry_t1) * BORROW_RATE_ANNUAL * hold_days / 252
                trades.append({
                    "bar":    i,
                    "time":   df.index[i],
                    "net":    gross - tx - borrow,
                    "reason": "STOP" if st else "EXIT",
                })
                pos = 0

        if pos == 0:
            if z < -entry_z: pos = 1
            elif z > entry_z: pos = -1
            if pos != 0:
                # ── Hurst drift guard ─────────────────────────────
                if hurst_filter is not None and spread_daily is not None:
                    ts = df.index[i]
                    d_prev = (ts - pd.Timedelta(days=1)).normalize()
                    h_tail_daily = spread_daily.loc[:d_prev].tail(HURST_ENTRY_WINDOW - 1)
                    h_tail = pd.concat([h_tail_daily, pd.Series({ts: df["spread"].iloc[i]})])
                    
                    h_blocked, _ = hurst_filter.should_block(h_tail, ts)
                    if h_blocked:
                        pos = 0
                        continue
                entry_spread = s; entry_t1 = p1; entry_t2 = p2; entry_bar = i
                entry_sma = df["spread_mean"].iloc[i]
                entry_std = df["spread_std"].iloc[i]

    return trades


def build_signals(closes: pd.DataFrame, t1: str, t2: str,
                  beta: float, half_life: float) -> pd.DataFrame:
    spread = closes[t1] - beta * closes[t2]
    window = max(20, min(int(half_life), 200))
    spread_mean = spread.rolling(window).mean()
    spread_std = spread.rolling(window).std()
    zscore = (spread - spread_mean) / spread_std
    return pd.DataFrame({
        f"{t1}_close": closes[t1],
        f"{t2}_close": closes[t2],
        "spread": spread, "zscore": zscore,
        "spread_mean": spread_mean, "spread_std": spread_std
    }).dropna().between_time(SIGNAL_START, RTH_END)


def metrics(trades: list[dict], days: float) -> dict:
    if not trades:
        return {"n": 0, "wr": 0, "sharpe": 0, "pnl": 0, "dd": 0}
    pnl   = np.array([t["net"] for t in trades])
    curve = np.cumsum(pnl)
    wr    = float((pnl > 0).mean() * 100)
    tpy   = len(pnl) / max(days / 365.25, 0.01)
    sh    = float(pnl.mean() / pnl.std() * np.sqrt(tpy)) if pnl.std() > 0 else 0.0
    dd    = float((curve - np.maximum.accumulate(curve)).min())
    return {"n": len(pnl), "wr": round(wr, 1), "sharpe": round(sh, 2),
            "pnl": round(float(pnl.sum()), 4), "dd": round(dd, 4)}


# ── Load data ─────────────────────────────────────────────────────────────────

closes_all = load_closes()
daily_all = load_daily()
pairs      = pd.read_csv(DATA_DIR / "pairs_selected.csv")
vix_daily  = load_vix_daily()

if pairs.empty:
    print("pairs_selected.csv is empty — run step2_pairs.py first")  # non-fatal

print(f"Stress test  |  {len(pairs)} pairs  |  "
      f"{closes_all.index[0].date()} → {closes_all.index[-1].date()}")
print(f"VIX data:    {'available' if vix_daily is not None else 'not found'}\n")

print("=" * 70)
print("WALK-FORWARD TEST  (Dynamic Beta)")
print("=" * 70)

wf_rows = []

for _, row in pairs.iterrows():
    pair_name = row["pair"]
    t1, t2    = pair_name.split("-")
    if t1 not in closes_all.columns or t2 not in closes_all.columns:
        continue
        
    start = closes_all.index[0]
    end   = closes_all.index[-1]
    w     = pd.DateOffset(months=WINDOW_MONTHS)
    step  = pd.DateOffset(months=STEP_MONTHS)
    cur   = start

    windows = []
    hf = HurstFilter()
    
    while cur + w <= end + pd.DateOffset(days=1):
        win_end = cur + w
        
        # Dynamic beta calculation over trailing 3 years
        train_start = cur - pd.DateOffset(years=3)
        daily_train = daily_all[(daily_all.index >= train_start) & (daily_all.index < cur)]
        
        dyn_beta = float(row.get("beta_daily", row["beta"]))
        dyn_hl = float(row["half_life_bars"])
        
        if len(daily_train) >= 100:
            is_coint, b = check_coint_johansen(daily_train, t1, t2, crit_level=0.90)
            if is_coint and b is not None and b > 0:
                dyn_beta = b
                spread_train = daily_train[t1] - dyn_beta * daily_train[t2]
                hl = compute_half_life(spread_train) * BARS_PER_DAY
                if 20 <= hl <= 500:
                    dyn_hl = hl

        # Build signals for the whole history (fast enough) to get correct rolling Z
        df_sig = build_signals(closes_all, t1, t2, dyn_beta, dyn_hl)
        win_df = df_sig[(df_sig.index >= cur) & (df_sig.index < win_end)]
        
        if len(win_df) > 200:
            # Daily spread using dynamic beta for Hurst
            spread_daily = (daily_all[t1] - dyn_beta * daily_all[t2]).dropna()
            windows.append((cur.date(), (win_end - pd.Timedelta(days=1)).date(), win_df, dyn_beta, spread_daily))
            
        cur += step

    if not windows:
        continue

    print(f"\n  {pair_name}  ({len(windows)} windows)")
    print(f"  {'Window':<24} {'Trades':>7} {'WR':>6} {'Sharpe':>8} {'P&L':>10}")
    print(f"  {'─'*58}")

    for w_start, w_end, win_df, dyn_beta, spread_daily in windows:
        days_w = (pd.Timestamp(w_end) - pd.Timestamp(w_start)).days
        trades = run_backtest(win_df, t1, t2, dyn_beta, ENTRY_Z, EXIT_Z, STOP_Z, hf, spread_daily)
        m      = metrics(trades, days_w)
        flag   = "  ◄ GOOD" if m["sharpe"] > 1 else ("  ✗ BAD" if m["sharpe"] < -1 else "")
        print(f"  {str(w_start)} → {str(w_end)}  "
              f"{m['n']:>7}  {m['wr']:>5.1f}%  {m['sharpe']:>8.2f}  "
              f"{m['pnl']:>+10.4f}{flag}")
        wf_rows.append({"pair": pair_name, "window_start": str(w_start),
                        "window_end": str(w_end), **m})

wf_df = pd.DataFrame(wf_rows)
wf_df.to_csv(DATA_DIR / "stress_walkforward.csv", index=False)

# Walk-forward chart
if not wf_df.empty:
    fig, axes = plt.subplots(len(pairs), 1,
                             figsize=(13, 4 * len(pairs)), squeeze=False)
    for ax_row, (_, row) in zip(axes, pairs.iterrows()):
        pair_name = row["pair"]
        ax  = ax_row[0]
        sub = wf_df[wf_df["pair"] == pair_name].reset_index(drop=True)
        if sub.empty:
            ax.set_visible(False)
            continue
        colors = ["green" if s > 0 else "red" for s in sub["sharpe"]]
        ax.bar(range(len(sub)), sub["sharpe"], color=colors, alpha=0.8)
        ax.axhline(0, color="black", lw=0.8)
        ax.axhline(1, color="green", lw=1, ls="--", alpha=0.5)
        ax.axhline(-1, color="red",  lw=1, ls="--", alpha=0.5)
        ax.set_xticks(range(len(sub)))
        ax.set_xticklabels([r["window_start"][:7] for _, r in sub.iterrows()],
                           rotation=30, fontsize=8)
        ax.set_title(f"{pair_name}  —  Sharpe per {WINDOW_MONTHS}-month window", fontweight="bold")
        ax.set_ylabel("Sharpe")

    plt.suptitle("Walk-Forward Consistency", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "stress_walkforward.png", dpi=150)
    plt.show()
    plt.close()
    print(f"\nWalk-forward chart saved → {OUTPUT_DIR / 'stress_walkforward.png'}")


# ═══════════════════════════════════════════════════════════════════════
# 2. VIX REGIME BREAKDOWN
# ═══════════════════════════════════════════════════════════════════════

print(f"\n{'='*70}")
print("VIX REGIME BREAKDOWN")
print(f"{'='*70}")

if vix_daily is not None:
    # Map VIX regime to intraday bars
    vix_tz = vix_daily.copy()
    if vix_tz.index.tz is None:
        vix_tz.index = vix_tz.index.tz_localize("UTC").tz_convert("US/Eastern")
    else:
        vix_tz.index = vix_tz.index.tz_convert("US/Eastern")

    def vix_regime(date) -> str:
        d = pd.Timestamp(date).tz_localize("US/Eastern") if date.tzinfo is None else date
        candidates = vix_tz[vix_tz.index <= d]
        if candidates.empty:
            return "normal"
        v = candidates.iloc[-1]
        if v < VIX_LOW:   return "calm"
        if v > VIX_HIGH:  return "elevated"
        return "normal"

    regime_labels = ["calm", "normal", "elevated"]

    for _, row in pairs.iterrows():
        pair_name = row["pair"]
        t1, t2 = pair_name.split("-")
        beta = float(row.get("beta_daily", row["beta"]))
        half_life = float(row["half_life_bars"])
        if t1 not in closes_all.columns or t2 not in closes_all.columns:
            continue
        df_sig = build_signals(closes_all, t1, t2, beta, half_life)
        spread_daily = (daily_all[t1] - beta * daily_all[t2]).dropna()
        print(f"\n  {pair_name}")
        print(f"  {'Regime':<12} {'Trades':>7} {'WR':>6} {'Sharpe':>8} {'P&L':>10} {'VIX range'}")
        print(f"  {'─'*58}")
        for regime in regime_labels:
            mask = df_sig.index.map(lambda ts: vix_regime(ts) == regime)
            sub  = df_sig[mask]
            if len(sub) < 50:
                print(f"  {regime:<12}  (not enough data)")
                continue
            days = (sub.index[-1] - sub.index[0]).days
            hf   = HurstFilter()
            tr   = run_backtest(sub, t1, t2, beta, ENTRY_Z, EXIT_Z, STOP_Z, hf)
            m    = metrics(tr, days)
            vix_range = {"calm": f"< {VIX_LOW}",
                         "normal": f"{VIX_LOW}–{VIX_HIGH}",
                         "elevated": f"> {VIX_HIGH}"}[regime]
            print(f"  {regime:<12}  {m['n']:>7}  {m['wr']:>5.1f}%  "
                  f"{m['sharpe']:>8.2f}  {m['pnl']:>+10.4f}  VIX {vix_range}")
else:
    print("  VIX data not available — run step7_iv.py first")


# ═══════════════════════════════════════════════════════════════════════
# 3. "TRADE TODAY?" DECISION
# ═══════════════════════════════════════════════════════════════════════

print(f"\n{'='*70}")
print("TRADE TODAY? — LIVE SIGNAL")
print(f"{'='*70}")

# Load supporting data
mc_path = DATA_DIR / "ou_montecarlo_summary.csv"
mc_conf: dict[str, float] = {}
if mc_path.exists():
    mc_df = pd.read_csv(mc_path)
    for _, r in mc_df.iterrows():
        mc_conf[r["pair"]] = float(r["win_rate"]) / 100

regimes_path = DATA_DIR / "regimes.csv"
current_regime: dict[str, str] = {}
if regimes_path.exists():
    reg_df = pd.read_csv(regimes_path, index_col=0, parse_dates=True)
    for col in reg_df.columns:
        last_val = reg_df[col].dropna().iloc[-1] if not reg_df[col].dropna().empty else 0
        current_regime[col] = "volatile" if last_val == 1 else "normal"

current_vix = None
if vix_daily is not None and len(vix_daily) > 0:
    current_vix = float(vix_daily.iloc[-1])

# Walk-forward consistency score: fraction of windows with Sharpe > 0
wf_consistency: dict[str, float] = {}
if not wf_df.empty:
    for pair_name in pairs["pair"]:
        sub = wf_df[wf_df["pair"] == pair_name]
        if len(sub) > 0:
            wf_consistency[pair_name] = float((sub["sharpe"] > 0).mean())

print(f"\n  Current VIX: {f'{current_vix:.1f}' if current_vix else 'n/a'}")
print()
print(f"  {'Pair':<12} {'Regime':<10} {'MC_conf':>8} {'WF_ok%':>7} "
      f"{'Recent_corr':>12} {'Signal':<10} {'Reason'}")
print(f"  {'─'*80}")

signals_today = []

for _, row in pairs.iterrows():
    pair_name = row["pair"]
    regime    = current_regime.get(pair_name, "unknown")
    mc        = mc_conf.get(pair_name, 1.0)
    wf_ok     = wf_consistency.get(pair_name, 0.5)
    rc        = float(row.get("recent_corr_120d", 1.0))

    # Score each dimension
    issues    = []
    score     = 0

    if mc < 0.45:
        issues.append(f"MC={mc:.2f}<0.45")
    else:
        score += 1

    if wf_ok < 0.5:
        issues.append(f"WF_ok={wf_ok:.0%}<50%")
    else:
        score += 1

    if rc < 0.55:
        issues.append(f"rc120={rc:.2f}<0.55")
    else:
        score += 1

    if regime == "volatile":
        issues.append("volatile regime")
    else:
        score += 1

    if current_vix is not None and current_vix > VIX_HIGH:
        issues.append(f"VIX={current_vix:.0f}>{VIX_HIGH}")
    else:
        score += 1

    if score == 5:
        signal = "✅ GO"
    elif score >= 3:
        signal = "⚠️  CAUTION"
    else:
        signal = "🔴 STOP"

    reason = ", ".join(issues) if issues else "all clear"
    print(f"  {pair_name:<12} {regime:<10} {mc:>8.2f} {wf_ok:>6.0%}  "
          f"{rc:>11.3f}  {signal:<12}  {reason}")

    signals_today.append({
        "pair": pair_name, "regime": regime, "mc_conf": mc,
        "wf_consistency": wf_ok, "recent_corr": rc, "signal": signal,
        "issues": reason,
    })

pd.DataFrame(signals_today).to_csv(DATA_DIR / "stress_signal_today.csv", index=False)

# VIX context
if current_vix is not None:
    vix_regime_now = ("calm" if current_vix < VIX_LOW
                      else "elevated" if current_vix > VIX_HIGH
                      else "normal")
    print(f"\n  Market regime: VIX={current_vix:.1f} → {vix_regime_now.upper()}")

print(f"\nSaved → data/stress_signal_today.csv")
print(f"        data/stress_walkforward.csv")
print(f"        output/stress_walkforward.png")


### 5f — Z-Score Debug Plots


In [ ]:
# Fallback: load df_trades from CSV if not in memory (e.g. running this cell standalone)
if 'df_trades' not in dir() or df_trades is None:
    import pandas as pd
    from config import DATA_DIR
    df_trades = pd.read_csv(DATA_DIR / "trades.csv")
    df_trades["exit_time"]  = pd.to_datetime(df_trades["exit_time"],  utc=True)
    df_trades["entry_time"] = pd.to_datetime(df_trades["entry_time"], utc=True)
    print(f"Loaded {len(df_trades)} trades from CSV")
else:
    print(f"Using in-memory df_trades: {len(df_trades)} trades")

# ── Step 5f: Z-Score Debug Plots ─────────────────────────────────────────────
from step5f_debug_plot import plot_zscore_debug
from config import DATA_DIR, ENTRY_Z, EXIT_Z, STOP_Z

if 'pair_results' in dir() and pair_results:
    _opt_params_5f = {}
    _opt_path_5f = DATA_DIR / "optimal_params.csv"
    if _opt_path_5f.exists():
        _opt_df_5f = pd.read_csv(_opt_path_5f)
        for _, _r in _opt_df_5f.iterrows():
            _opt_params_5f[_r["pair"]] = (float(_r["entry_z"]), float(_r["exit_z"]), float(_r["stop_z"]))

    print("Generating Z-score debug plots ...")
    for _pair_name, _data in pair_results.items():
        _ez, _xz, _sz = _opt_params_5f.get(_pair_name, (ENTRY_Z, EXIT_Z, STOP_Z))
        _out = plot_zscore_debug(
            _data["signals"], _data["trades"],
            _pair_name, entry_z=_ez, exit_z=_xz, stop_z=_sz,
        )
        from IPython.display import Image, display as ipy_display
        ipy_display(Image(str(_out)))
else:
    print("pair_results not in memory — debug plots require Step 4 to have run in this session")
